# Oyster Mushroom Cluster Monitoring: fresh inference and segmentation evaluation

This notebook reloads the canonical data and both selected checkpoints, performs fresh forward passes on all 961 validation/test frames, and exports prediction records, segmentation metrics, matched-instance geometry results, figures, provenance, and integrity checks. It also retains the earlier integrated tracker-comparison diagnostics as part of the full research evidence record. The final frozen-tracker and growth-history results reported in the paper are evaluated separately by `tracking-and-growth-history.ipynb`, which consumes the cached YOLO predictions from this run.

The notebook starts a clean run by removing only its previous output directory under `/kaggle/working`; it does not reuse earlier prediction caches for a new inference pass. If a completed fresh-inference cache is reused to resume post-inference analysis after an interruption, the run manifest records that explicitly.

The test sequences were examined during earlier development. Recomputing the results establishes computational reproducibility, not an untouched independent test estimate.

The saved execution outputs in this repository may retain the earlier project-prefixed output paths and archive names. Those paths are preserved as historical run records; a new execution uses the current project output directory. Legacy input archive names remain accepted by the discovery code for compatibility.


In [1]:
from pathlib import Path
import shutil

# Leave as None for recursive discovery under /kaggle/input and /kaggle/working.
YOLO_RESULTS_ARCHIVE_PATH = None      # FungiCirno_YOLO26_Charisis_Full_Results.zip or extracted results folder
YOLO_CHECKPOINT_PATH = None           # selected_yolo26l_charisis.pt, if attached directly
MASKRCNN_MODEL_ROOT = None            # folder containing model_checkpoint.pth (+ optional JSON config files)
SOURCE_DATA_ROOT = None               # extracted Charisis root or parent containing the four sequence ZIPs
SPLIT_MANIFEST_PATH = None            # canonical split_manifest.csv, if attached directly
CONTINUITY_ZIP_PATH = None            # optional FungiCirno_Continuity_Pack_*.zip

ALLOW_GOOGLE_DRIVE_DOWNLOAD = True    # used only when source images/annotations are not attached
DRIVE_FOLDER_URL = "https://drive.google.com/drive/folders/1dDEM7CdYQaNKbWb5Qo2KNOdkB8F48c0L?usp=sharing"

# Exact provenance expected from the existing project artifacts.
EXPECTED_MANIFEST_SHA256 = "42e18712fdff1c2fb263de7006fdfdf5a62a2d981764a823ca707c3897d44cd6"
EXPECTED_YOLO_SHA256 = "7964816abe44fd635725fa0997aaf161ce8d7429c7f265116bf8e53c966cbb8b"
EXPECTED_MASKRCNN_SHA256 = "3aabb8c4a9ed6016c29b80008afe6dc1d4a1c454392bebaac3fc110c0c79f2d8"

# Reproduce the original inference settings. Keep Ultralytics pinned to the recorded version.
ULTRALYTICS_VERSION = "8.4.214"
YOLO_IMAGE_SIZE = 1024
YOLO_SCORE_FLOOR = 0.001
YOLO_NMS_IOU = 0.70
YOLO_NMS_MODE = None  # exact original Ultralytics setting from the supplied YOLO inference notebook
YOLO_MAX_DET = 1000
YOLO_RETINA_MASKS = True
MASKRCNN_SCORE_FLOOR = 0.001
MASKRCNN_MAX_DET = 100
MASKRCNN_MASK_THRESHOLD = 0.50
MASK_MATCH_IOU = 0.50
AP_MAX_DETS = 100
WARMUP_IMAGES_PER_SEQUENCE = 2

# Frozen tracker candidate settings from the completed retrospective 2x2 ablation.
# The high-confidence threshold is freshly selected from validation for this inference run.
TRACKER_ASSOCIATION_IOU = 0.10
TRACKER_LOW_CONFIDENCE_MIN = 0.60
TRACKER_MAX_GAP = 0
TRACKER_SPATIAL_DISTANCE_PX = 47.0
TRACKER_SPATIAL_MAX_COST = 0.80
SELECTED_TRACKING_VARIANT = "C_iou_plus_lowconf"  # mask-IoU + low-confidence recovery
VALIDATION_CADENCE_STRIDE = 8

# Fresh-run mode: remove prior outputs and rerun both models; do not reuse prediction caches.
CLEAN_PREVIOUS_OUTPUTS = True
REUSE_COMPLETED_FRESH_INFERENCE_IF_VALID = False

# Figure settings. Source thumbnails are omitted to keep the evidence bundle compact; source images are loaded on demand.
SAVE_SOURCE_THUMBNAILS = False
THUMBNAIL_WIDTH = 1600
THUMBNAIL_JPEG_QUALITY = 82
MAX_QUALITATIVE_FRAMES = 12
MAX_TRACK_GROWTH_FIGURES = 4
RANDOM_SEED = 42

WORK_DIR = Path("/kaggle/working/Oyster_Mushroom_Cluster_Monitoring_Independent_Evidence")
if CLEAN_PREVIOUS_OUTPUTS and WORK_DIR.exists():
    shutil.rmtree(WORK_DIR)
OUTPUT_DIR = WORK_DIR / "evidence"
CACHE_DIR = OUTPUT_DIR / "fresh_prediction_records"
FIGURE_DIR = OUTPUT_DIR / "figures"
THUMBNAIL_DIR = OUTPUT_DIR / "source_thumbnails"
for path in [WORK_DIR, OUTPUT_DIR, CACHE_DIR, FIGURE_DIR, THUMBNAIL_DIR]:
    path.mkdir(parents=True, exist_ok=True)

EXPECTED_SOURCE_IMAGE_COUNTS = {1: 2084, 2: 1175, 3: 2133, 4: 990}
EXPECTED_SPLIT_COUNTS = {"train": 1728, "val": 433, "test": 528}
EXPECTED_EVAL_BY_SEQUENCE = {1: 261, 2: 235, 3: 267, 4: 198}

## Environment

The runtime is pinned to the recorded Ultralytics version. The Kaggle-provided PyTorch, TorchVision, and CUDA builds are retained to avoid replacing a compatible GPU stack.

In [2]:
import importlib.util, subprocess, sys
from importlib.metadata import version, PackageNotFoundError

required = {
    "ultralytics": f"ultralytics=={ULTRALYTICS_VERSION}",
    "pycocotools": "pycocotools",
    "gdown": "gdown",
    "psutil": "psutil",
}
missing = [spec for module, spec in required.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
try:
    installed_ultralytics = version("ultralytics")
except PackageNotFoundError:
    installed_ultralytics = None
if installed_ultralytics != ULTRALYTICS_VERSION:
    # Do not let pip replace Kaggle's CUDA-matched PyTorch/TorchVision stack.
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "--force-reinstall", f"ultralytics=={ULTRALYTICS_VERSION}"])
# Do not reinstall PyTorch or TorchVision here: retain the Kaggle GPU runtime's compatible builds.


In [3]:
import os, re, io, gc, json, gzip, time, math, shutil, hashlib, zipfile, random, platform, warnings
from collections import defaultdict, Counter
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import cv2
import torch
import torchvision
import torch.nn as nn
import matplotlib.pyplot as plt
import psutil
import ultralytics
import gdown
from tqdm.auto import tqdm
from IPython.display import display
from PIL import Image
from scipy.optimize import linear_sum_assignment
from pycocotools import mask as mask_utils
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
from torchvision.models.detection import MaskRCNN
from torchvision.ops.feature_pyramid_network import FeaturePyramidNetwork, LastLevelMaxPool
from torchvision.transforms import functional as TVF
from torchvision.models.detection.transform import resize_boxes, GeneralizedRCNNTransform
from torchvision.models.detection.roi_heads import paste_masks_in_image
from ultralytics import YOLO

SEED = RANDOM_SEED
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE = 0 if torch.cuda.is_available() else "cpu"  # Ultralytics device selector
TORCH_DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
KAGGLE_INPUT = Path("/kaggle/input")

if not torch.cuda.is_available():
    raise RuntimeError("A Kaggle GPU accelerator is required for the full-resolution YOLO26-L and Mask R-CNN inference run.")
if ultralytics.__version__ != ULTRALYTICS_VERSION:
    raise RuntimeError(f"Ultralytics version mismatch: installed={ultralytics.__version__}, expected={ULTRALYTICS_VERSION}")

print("GPU:", torch.cuda.get_device_name(0))
print("GPU memory (GiB):", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
print("Python:", sys.version)
print("PyTorch:", torch.__version__, "| TorchVision:", torchvision.__version__, "| CUDA:", torch.version.cuda)
print("Ultralytics:", ultralytics.__version__, "| NumPy:", np.__version__, "| OpenCV:", cv2.__version__)

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
GPU: Tesla T4
GPU memory (GiB): 14.56
Python: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
PyTorch: 2.11.0+cu130 | TorchVision: 0.26.0+cu130 | CUDA: 13.0
Ultralytics: 8.4.214 | NumPy: 2.1.3 | OpenCV: 4.14.0


## Data and provenance

The loader searches the Kaggle input and working directories, verifies the canonical split manifest and model checkpoints, and reconstructs the four source sequences from their original archives and tracked COCO annotations. Explicit paths can be supplied in the configuration cell when automatic discovery is ambiguous.

In [4]:
def sha256_file(path, chunk_size=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for block in iter(lambda: handle.read(chunk_size), b""):
            digest.update(block)
    return digest.hexdigest()


def unique_path(paths, label):
    paths = sorted({Path(p).resolve() for p in paths})
    if len(paths) != 1:
        raise RuntimeError(f"Expected exactly one {label}; found {len(paths)}: {[str(p) for p in paths[:10]]}. Set its explicit path in the settings cell.")
    return paths[0]


# Preserve the project's established v5/v6 discovery convention: recurse through
# BOTH Kaggle input datasets and the working directory. Kaggle's attached
# extracted artifacts may appear under either root and may use the prefixed
# filename FungiCirno_split_manifest.csv.
KAGGLE_SEARCH_ROOTS = [p for p in [KAGGLE_INPUT, Path("/kaggle/working")] if p.exists()]


def recursive_files(names, roots=None):
    names = set(names)
    found = set()
    for root in (KAGGLE_SEARCH_ROOTS if roots is None else roots):
        if root.exists():
            for p in root.rglob("*"):
                if p.is_file() and p.name in names:
                    found.add(p.resolve())
    return sorted(found, key=lambda p: str(p))


def safe_zip_extract_member(archive_path, member_name, destination):
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive_path) as zf:
        bad = zf.testzip()
        if bad:
            raise RuntimeError(f"Corrupt ZIP member {bad} in {archive_path}")
        matches = [n for n in zf.namelist() if n.endswith(member_name)]
        if len(matches) != 1:
            raise RuntimeError(f"Expected exactly one archive member ending {member_name!r}; found {matches[:8]}")
        destination.write_bytes(zf.read(matches[0]))
    return destination


def locate_optional_zip(explicit, basename):
    if explicit:
        p = Path(explicit).expanduser()
        if not p.is_file():
            raise FileNotFoundError(p)
        return p.resolve()
    matches = []
    for root in KAGGLE_SEARCH_ROOTS:
        matches.extend(p.resolve() for p in root.rglob(basename) if p.is_file())
    matches = sorted(set(matches), key=lambda p: str(p))
    if len(matches) > 1:
        raise RuntimeError(f"Multiple {basename} files found: {[str(p) for p in matches[:8]]}")
    return matches[0] if matches else None


# Canonical manifest resolution. Prefer an exact-hash standalone CSV anywhere
# under /kaggle/input or /kaggle/working; accept both historical filenames.
# If no standalone copy matches, inspect the continuity pack / starter pack ZIP.
if SPLIT_MANIFEST_PATH:
    MANIFEST_PATH = Path(SPLIT_MANIFEST_PATH).expanduser().resolve()
    if not MANIFEST_PATH.is_file():
        raise FileNotFoundError(MANIFEST_PATH)
else:
    manifest_hits = recursive_files({"split_manifest.csv", "FungiCirno_split_manifest.csv"})
    manifest_matches = [(p, sha256_file(p)) for p in manifest_hits]
    canonical_direct = [p for p, digest in manifest_matches if digest == EXPECTED_MANIFEST_SHA256]

    if canonical_direct:
        # Multiple byte-identical copies are harmless (e.g. a standalone CSV
        # plus an extracted starter pack). Select deterministically and record all.
        MANIFEST_PATH = sorted(canonical_direct, key=lambda p: str(p))[0]
        if len(canonical_direct) > 1:
            print("Found multiple copies of the same canonical manifest; using:", MANIFEST_PATH)
            print("Other identical copies:", [str(p) for p in sorted(canonical_direct, key=lambda p: str(p))[1:]])
    else:
        candidate_zips = []
        if CONTINUITY_ZIP_PATH:
            explicit_zip = Path(CONTINUITY_ZIP_PATH).expanduser().resolve()
            if not explicit_zip.is_file():
                raise FileNotFoundError(explicit_zip)
            candidate_zips = [explicit_zip]
        else:
            zip_names = {
                "FungiCirno_Continuity_Pack_2026-11-17(1).zip",
                "FungiCirno_Master_Kaggle_Starter_Pack.zip",
                "FungiCirno_YOLO26_Charisis_Full_Results.zip",
            }
            candidate_zips = recursive_files(zip_names)
            # Also accept later continuity-pack version suffixes.
            for root in KAGGLE_SEARCH_ROOTS:
                candidate_zips.extend(p.resolve() for p in root.rglob("FungiCirno_Continuity_Pack*.zip") if p.is_file())
            candidate_zips = sorted(set(candidate_zips), key=lambda p: str(p))

        archive_matches = []
        archive_errors = []
        for zp in candidate_zips:
            try:
                with zipfile.ZipFile(zp) as zf:
                    bad = zf.testzip()
                    if bad:
                        archive_errors.append(f"{zp}: corrupt member {bad}")
                        continue
                    members = [n for n in zf.namelist() if n.replace("\\", "/").endswith(("split_manifest.csv", "FungiCirno_split_manifest.csv"))]
                    for member in members:
                        member_bytes = zf.read(member)
                        digest = hashlib.sha256(member_bytes).hexdigest()
                        if digest == EXPECTED_MANIFEST_SHA256:
                            archive_matches.append((zp, member, member_bytes))
            except Exception as exc:
                archive_errors.append(f"{zp}: {type(exc).__name__}: {exc}")

        if not archive_matches:
            details = {
                "search_roots": [str(p) for p in KAGGLE_SEARCH_ROOTS],
                "standalone_manifest_candidates": [(str(p), h) for p, h in manifest_matches],
                "archive_candidates": [str(p) for p in candidate_zips],
                "archive_errors": archive_errors,
            }
            raise FileNotFoundError(
                "Canonical split manifest not found. The notebook searched recursively under "
                "/kaggle/input AND /kaggle/working for split_manifest.csv / "
                "FungiCirno_split_manifest.csv, then inspected the attached continuity/starter ZIPs. "
                "Attach the canonical standalone CSV or the starter/continuity ZIP, or set "
                f"SPLIT_MANIFEST_PATH. Discovery details: {json.dumps(details, indent=2)}"
            )

        # Every accepted archive member has the expected hash, so duplicate copies
        # are content-identical. Extract one deterministically.
        archive_matches.sort(key=lambda item: (str(item[0]), item[1]))
        zp, member, member_bytes = archive_matches[0]
        MANIFEST_PATH = WORK_DIR / "split_manifest.csv"
        MANIFEST_PATH.parent.mkdir(parents=True, exist_ok=True)
        MANIFEST_PATH.write_bytes(member_bytes)
        print("Canonical manifest extracted from:", zp, "member:", member)
        if len(archive_matches) > 1:
            print("Additional identical canonical manifest copies found:", len(archive_matches) - 1)

MANIFEST_SHA256 = sha256_file(MANIFEST_PATH)
if MANIFEST_SHA256 != EXPECTED_MANIFEST_SHA256:
    raise RuntimeError(f"Canonical split manifest hash mismatch: {MANIFEST_SHA256}")
manifest = pd.read_csv(MANIFEST_PATH)
required_cols = {"split", "sequence", "frame_idx", "image_id", "file_name", "annotations"}
if not required_cols.issubset(manifest.columns):
    raise RuntimeError(f"Manifest missing required columns: {sorted(required_cols - set(manifest.columns))}")
manifest["split"] = manifest["split"].astype(str).str.lower()
for c in ["sequence", "frame_idx", "image_id", "annotations"]:
    manifest[c] = pd.to_numeric(manifest[c], errors="raise").astype(int)
if manifest.groupby("split").size().to_dict() != EXPECTED_SPLIT_COUNTS:
    raise RuntimeError(f"Unexpected manifest split counts: {manifest.groupby('split').size().to_dict()}")
eval_manifest = manifest[manifest.split.isin(["val", "test"])].sort_values(["sequence", "frame_idx"]).reset_index(drop=True)
if len(eval_manifest) != 961:
    raise RuntimeError(f"Expected 961 evaluation frames; got {len(eval_manifest)}")
if eval_manifest.groupby("sequence").size().to_dict() != EXPECTED_EVAL_BY_SEQUENCE:
    raise RuntimeError(f"Unexpected evaluation sequence counts: {eval_manifest.groupby('sequence').size().to_dict()}")

# YOLO checkpoint: exact fine-tuned checkpoint only; search the same established
# input/working roots, then the full results archive. Never substitute pretrained weights.
YOLO_RESULTS_ARCHIVE = locate_optional_zip(YOLO_RESULTS_ARCHIVE_PATH, "FungiCirno_YOLO26_Charisis_Full_Results.zip")
if YOLO_CHECKPOINT_PATH:
    YOLO_MODEL_PATH = Path(YOLO_CHECKPOINT_PATH).expanduser().resolve()
    if not YOLO_MODEL_PATH.is_file():
        raise FileNotFoundError(YOLO_MODEL_PATH)
else:
    direct_hits = recursive_files({"selected_yolo26l_charisis.pt"})
    if len(direct_hits) == 1:
        YOLO_MODEL_PATH = direct_hits[0]
    elif len(direct_hits) > 1:
        YOLO_MODEL_PATH = unique_path(direct_hits, "fine-tuned YOLO26 checkpoint")
    elif YOLO_RESULTS_ARCHIVE is not None:
        YOLO_MODEL_PATH = safe_zip_extract_member(
            YOLO_RESULTS_ARCHIVE,
            "yolo26_charisis_artifacts/selected_yolo26l_charisis.pt",
            WORK_DIR / "model_artifacts" / "selected_yolo26l_charisis.pt",
        )
    else:
        raise FileNotFoundError("Attach the full YOLO training-results archive or the exact selected_yolo26l_charisis.pt file.")

YOLO_SHA256 = sha256_file(YOLO_MODEL_PATH)
if YOLO_SHA256 != EXPECTED_YOLO_SHA256:
    raise RuntimeError(f"YOLO checkpoint SHA-256 mismatch. Expected {EXPECTED_YOLO_SHA256}, got {YOLO_SHA256}. Do not substitute another checkpoint without updating the paper's provenance.")

# Mask R-CNN checkpoint: exact previously evaluated TorchVision adaptation.
if MASKRCNN_MODEL_ROOT:
    mr_root = Path(MASKRCNN_MODEL_ROOT).expanduser().resolve()
    if not mr_root.exists():
        raise FileNotFoundError(mr_root)
    mr_hits = list(mr_root.rglob("model_checkpoint.pth")) if mr_root.is_dir() else [mr_root]
else:
    mr_hits = recursive_files({"model_checkpoint.pth"})
MASKRCNN_CHECKPOINT_PATH = unique_path(mr_hits, "Mask R-CNN model_checkpoint.pth")
MASKRCNN_SHA256 = sha256_file(MASKRCNN_CHECKPOINT_PATH)
if MASKRCNN_SHA256 != EXPECTED_MASKRCNN_SHA256:
    raise RuntimeError(f"Mask R-CNN checkpoint SHA-256 mismatch. Expected {EXPECTED_MASKRCNN_SHA256}, got {MASKRCNN_SHA256}.")
MASKRCNN_BUNDLE_ROOT = MASKRCNN_CHECKPOINT_PATH.parent

def locate_optional_metadata(root, basename):
    hits = sorted((p.resolve() for p in root.rglob(basename) if p.is_file()), key=lambda p: str(p))
    if len(hits) > 1:
        raise RuntimeError(
            f"Multiple {basename} files found beside the selected Mask R-CNN checkpoint: "
            f"{[str(p) for p in hits[:8]]}. Set MASKRCNN_MODEL_ROOT to the exact model bundle."
        )
    return hits[0] if hits else None

MASKRCNN_CONFIG_PATH = locate_optional_metadata(MASKRCNN_BUNDLE_ROOT, "model_config.json")
MASKRCNN_CLASS_MAP_PATH = locate_optional_metadata(MASKRCNN_BUNDLE_ROOT, "class_mapping.json")

print("Kaggle discovery roots:", [str(p) for p in KAGGLE_SEARCH_ROOTS])
print("Manifest:", MANIFEST_PATH, "SHA-256:", MANIFEST_SHA256)
print("YOLO checkpoint:", YOLO_MODEL_PATH, "SHA-256:", YOLO_SHA256, "size MiB:", round(YOLO_MODEL_PATH.stat().st_size/2**20, 2))
print("Mask R-CNN checkpoint:", MASKRCNN_CHECKPOINT_PATH, "SHA-256:", MASKRCNN_SHA256, "size MiB:", round(MASKRCNN_CHECKPOINT_PATH.stat().st_size/2**20, 2))
display(eval_manifest.groupby(["sequence", "split"]).size().rename("frames").to_frame())

Kaggle discovery roots: ['/kaggle/input', '/kaggle/working']
Manifest: /kaggle/input/datasets/hnamedu/charisis-full/FungiCirno_YOLO26_Charisis_Full_Results/yolo26_charisis_artifacts/split_manifest.csv SHA-256: 42e18712fdff1c2fb263de7006fdfdf5a62a2d981764a823ca707c3897d44cd6
YOLO checkpoint: /kaggle/input/datasets/hnamedu/charisis-full/FungiCirno_YOLO26_Charisis_Full_Results/yolo26_charisis_artifacts/selected_yolo26l_charisis.pt SHA-256: 7964816abe44fd635725fa0997aaf161ce8d7429c7f265116bf8e53c966cbb8b size MiB: 180.37
Mask R-CNN checkpoint: /kaggle/input/datasets/hnamedu/charisis-full/model_bundle/model_checkpoint.pth SHA-256: 3aabb8c4a9ed6016c29b80008afe6dc1d4a1c454392bebaac3fc110c0c79f2d8 size MiB: 545.97


,,frames
sequence,split,
1,test,261
2,val,235
3,test,267
4,val,198


In [5]:
IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png", ".tif", ".tiff", ".bmp"}
SEQUENCE_ARCHIVES = {1: "1st_timelapse.zip", 2: "2nd_timelapse.zip", 3: "3rd_timelapse.zip", 4: "4th_timelapse.zip"}
EXPECTED_CATEGORY_REFERENCE = "mature_cluster"
SOURCE_WORK_ROOT = WORK_DIR / "source_sequences"
SOURCE_WORK_ROOT.mkdir(parents=True, exist_ok=True)


def natural_key(value):
    return [int(part) if part.isdigit() else part.casefold() for part in re.split(r"(\d+)", str(value))]


def infer_sequence(text):
    s = str(text).lower().replace("\\", "/")
    patterns = {
        1: [r"(?<![a-z0-9])1st(?![a-z0-9])", r"(?<![a-z0-9])first(?![a-z0-9])", r"2?lapse[_ -]?1(?!\d)", r"seq(?:uence)?[_ -]?1(?!\d)"],
        2: [r"(?<![a-z0-9])2nd(?![a-z0-9])", r"(?<![a-z0-9])second(?![a-z0-9])", r"lapse[_ -]?2(?!\d)", r"seq(?:uence)?[_ -]?2(?!\d)"],
        3: [r"(?<![a-z0-9])3rd(?![a-z0-9])", r"(?<![a-z0-9])third(?![a-z0-9])", r"lapse[_ -]?3(?!\d)", r"seq(?:uence)?[_ -]?3(?!\d)"],
        4: [r"(?<![a-z0-9])4th(?![a-z0-9])", r"(?<![a-z0-9])fourth(?![a-z0-9])", r"lapse[_ -]?4(?!\d)", r"seq(?:uence)?[_ -]?4(?!\d)"],
    }
    for seq, regs in patterns.items():
        if any(re.search(p, s) for p in regs): return seq
    return None


def safe_extract_sequence(zip_path, destination):
    destination = Path(destination)
    marker = destination / ".extraction_complete"
    archive_sha = sha256_file(zip_path)
    if marker.exists() and marker.read_text(encoding="utf-8").strip() == archive_sha:
        return destination
    if destination.exists(): shutil.rmtree(destination)
    destination.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path) as zf:
        bad = zf.testzip()
        if bad: raise RuntimeError(f"Corrupt source archive member: {bad}")
        zf.extractall(destination)
    for _ in range(3):
        nested = [p for p in destination.rglob("*.zip") if p.is_file()]
        if not nested: break
        for nested_zip in nested:
            target = nested_zip.parent / nested_zip.stem
            target.mkdir(parents=True, exist_ok=True)
            with zipfile.ZipFile(nested_zip) as zf:
                bad = zf.testzip()
                if bad: raise RuntimeError(f"Corrupt nested source archive member: {bad}")
                zf.extractall(target)
            nested_zip.unlink()
    marker.write_text(archive_sha, encoding="utf-8")
    return destination


def locate_attached_sequence_root(seq):
    token = Path(SEQUENCE_ARCHIVES[seq]).stem
    if SOURCE_DATA_ROOT:
        root = Path(SOURCE_DATA_ROOT).expanduser().resolve()
        if not root.exists(): raise FileNotFoundError(root)
        tracked = [p for p in root.rglob("*_annotations_tracked.json") if infer_sequence(p) == seq or token in p.name.lower()]
        if len(tracked) == 1: return tracked[0].parent.parent, "attached_extracted", None
        archives = list(root.rglob(SEQUENCE_ARCHIVES[seq]))
        if len(archives) == 1:
            out = SOURCE_WORK_ROOT / f"sequence_{seq}"
            return safe_extract_sequence(archives[0], out), "attached_archive", sha256_file(archives[0])
        if len(archives) > 1: raise RuntimeError(f"Multiple source archives for sequence {seq}: {archives[:8]}")
    tracked = []
    archives = []
    for search_root in KAGGLE_SEARCH_ROOTS:
        tracked.extend(p for p in search_root.rglob("*_annotations_tracked.json") if infer_sequence(p) == seq or token in p.name.lower())
        archives.extend(search_root.rglob(SEQUENCE_ARCHIVES[seq]))
    tracked = sorted(set(p.resolve() for p in tracked), key=lambda p: str(p))
    archives = sorted(set(p.resolve() for p in archives), key=lambda p: str(p))
    if len(tracked) == 1:
        return tracked[0].parent.parent, "attached_extracted", None
    if len(tracked) > 1:
        raise RuntimeError(f"Multiple tracked annotation files for sequence {seq}: {tracked[:8]}")
    if len(archives) == 1:
        out = SOURCE_WORK_ROOT / f"sequence_{seq}"
        return safe_extract_sequence(archives[0], out), "attached_archive", sha256_file(archives[0])
    if len(archives) > 1:
        raise RuntimeError(f"Multiple source archives for sequence {seq}: {archives[:8]}")
    return None, None, None

_DRIVE_ITEMS = None

def get_drive_items():
    global _DRIVE_ITEMS
    if _DRIVE_ITEMS is not None: return _DRIVE_ITEMS
    if not ALLOW_GOOGLE_DRIVE_DOWNLOAD:
        raise FileNotFoundError("Source sequence archives are not attached and Google Drive download is disabled.")
    entries = gdown.download_folder(url=DRIVE_FOLDER_URL, quiet=True, skip_download=True)
    if not entries: raise RuntimeError("Cannot list the shared Charisis Drive folder. Check Kaggle Internet and Drive sharing permissions.")
    result = {}
    for item in entries:
        name = Path(str(item.path)).name
        if name in SEQUENCE_ARCHIVES.values():
            if name in result: raise RuntimeError(f"Duplicate Drive archive listing entry: {name}")
            result[name] = item
    missing = set(SEQUENCE_ARCHIVES.values()) - set(result)
    if missing: raise RuntimeError(f"Shared Drive folder is missing: {sorted(missing)}")
    _DRIVE_ITEMS = result
    return result


def get_sequence_root(seq):
    root, mode, archive_sha = locate_attached_sequence_root(seq)
    if root is not None:
        return root, mode, archive_sha
    item = get_drive_items()[SEQUENCE_ARCHIVES[seq]]
    archive_path = SOURCE_WORK_ROOT / SEQUENCE_ARCHIVES[seq]
    if not archive_path.exists() or not zipfile.is_zipfile(archive_path):
        got = gdown.download(id=item.id, output=str(archive_path), quiet=False, resume=True)
        if got is None or not zipfile.is_zipfile(archive_path):
            raise RuntimeError(f"Could not download a valid source archive for sequence {seq}.")
    archive_sha = sha256_file(archive_path)
    out = SOURCE_WORK_ROOT / f"sequence_{seq}"
    root = safe_extract_sequence(archive_path, out)
    return root, "google_drive", archive_sha


def build_image_index(root):
    index = defaultdict(list)
    for p in Path(root).rglob("*"):
        if p.is_file() and p.suffix.lower() in IMAGE_SUFFIXES:
            index[p.name.casefold()].append(p.resolve())
    return index


def resolve_image_path(file_name, json_path, root, image_index, seq):
    raw = str(file_name).replace("\\", "/").lstrip("./")
    base = Path(raw).name
    candidates = [Path(json_path).parent / raw, Path(root) / raw,
                  Path(json_path).parent / base, Path(root) / base]
    for p in candidates:
        if p.is_file():
            inferred = infer_sequence(p)
            if inferred is None or inferred == seq: return p.resolve()
    matches = image_index.get(base.casefold(), [])
    seq_matches = [p for p in matches if infer_sequence(p) == seq]
    if len(seq_matches) == 1: return seq_matches[0]
    if len(seq_matches) > 1: raise RuntimeError(f"Ambiguous image path for {file_name}: {seq_matches[:5]}")
    if len(matches) == 1:
        inferred = infer_sequence(matches[0])
        if inferred is not None and inferred != seq:
            raise RuntimeError(
                f"Resolved image {matches[0]} for sequence {seq}, but its path identifies "
                f"sequence {inferred}. Refusing a cross-sequence image substitution."
            )
        return matches[0]
    raise FileNotFoundError(f"Could not uniquely resolve source image {file_name!r} in sequence {seq}.")

In [6]:
def annotation_to_rle(annotation, height, width):
    seg = annotation.get("segmentation")
    if isinstance(seg, list):
        polygons = [p for p in seg if isinstance(p, list) and len(p) >= 6]
        if not polygons: raise ValueError(f"Annotation {annotation.get('id')} contains no valid polygon segments.")
        rles = mask_utils.frPyObjects(polygons, int(height), int(width))
        return mask_utils.merge(rles) if isinstance(rles, list) else rles
    if isinstance(seg, dict):
        rle = dict(seg)
        if isinstance(rle.get("counts"), list):
            rle = mask_utils.frPyObjects(rle, int(height), int(width))
        elif isinstance(rle.get("counts"), str):
            rle["counts"] = rle["counts"].encode("ascii")
        return mask_utils.merge(rle) if isinstance(rle, list) else rle
    raise ValueError(f"Unsupported or missing segmentation for annotation {annotation.get('id')}")


def rle_to_json(rle):
    counts = rle["counts"].decode("ascii") if isinstance(rle["counts"], bytes) else str(rle["counts"])
    return {"size": [int(x) for x in rle["size"]], "counts": counts}


def tight_geometry(binary_mask, include_centroid=True):
    mask = np.asarray(binary_mask, dtype=np.uint8)
    area = int(cv2.countNonZero(mask))
    if area <= 0: return None
    x, y, w, h = cv2.boundingRect(mask)
    result = {"AREA_PX": area, "WIDTH_PX": int(w), "HEIGHT_PX": int(h),
              "BBOX_XYXY": [int(x), int(y), int(x+w), int(y+h)],
              "BBOX_XYWH": [int(x), int(y), int(w), int(h)]}
    if include_centroid:
        moments = cv2.moments(mask, binaryImage=True)
        result["CENTROID_X_PX"] = float(moments["m10"] / moments["m00"])
        result["CENTROID_Y_PX"] = float(moments["m01"] / moments["m00"])
    return result


def annotation_to_instance(annotation, height, width, local_idx, categories):
    rle = annotation_to_rle(annotation, height, width)
    binary = mask_utils.decode(rle).astype(np.uint8)
    if binary.shape != (height, width):
        raise RuntimeError(f"GT mask shape mismatch for annotation {annotation.get('id')}: {binary.shape} != {(height,width)}")
    geometry = tight_geometry(binary)
    if geometry is None: raise RuntimeError(f"Empty source GT segmentation: annotation {annotation.get('id')}")
    track_id = annotation.get("tracking_id")
    if track_id is None:
        raise RuntimeError(f"Missing tracking_id on source annotation {annotation.get('id')}.")
    category_id = int(annotation["category_id"])
    return {
        "GT_FRAME_INSTANCE_ID": int(local_idx),
        "COCO_ANNOTATION_ID": int(annotation["id"]),
        "GT_TRACK_ID": str(track_id),
        "ORIGINAL_CATEGORY_ID": category_id,
        "ORIGINAL_CATEGORY_NAME": categories.get(category_id, str(category_id)),
        "IS_CROWD": bool(annotation.get("iscrowd", 0)),
        "COCO_BBOX_XYWH": [float(v) for v in annotation.get("bbox", [])],
        **geometry,
        "MASK_RLE": rle_to_json(rle),
    }


def load_sequence_bundle(seq, manifest_df):
    root, source_mode, source_archive_sha = get_sequence_root(seq)
    tracked_candidates = [p for p in Path(root).rglob("*_annotations_tracked.json")
                          if infer_sequence(p) == seq or SEQUENCE_ARCHIVES[seq].replace(".zip", "") in p.name.lower()]
    if len(tracked_candidates) != 1:
        raise RuntimeError(f"Sequence {seq}: expected exactly one *_annotations_tracked.json, found {tracked_candidates[:8]}")
    tracked_path = tracked_candidates[0]
    coco = json.loads(tracked_path.read_text(encoding="utf-8"))
    if not {"images", "annotations", "categories"}.issubset(coco):
        raise RuntimeError(f"Tracked annotation file is not a COCO document: {tracked_path}")

    categories = {int(c["id"]): str(c.get("name", c["id"])) for c in coco["categories"]}
    # Full-source identities span every annotated frame in this sequence, not only eval frames.
    source_track_ids = sorted({str(a["tracking_id"]) for a in coco["annotations"]})
    anns_by_image = defaultdict(list)
    ann_ids = set()
    for ann in coco["annotations"]:
        ann_id = int(ann["id"])
        if ann_id in ann_ids: raise RuntimeError(f"Sequence {seq}: duplicate COCO annotation ID {ann_id}")
        ann_ids.add(ann_id)
        if ann.get("tracking_id") is None: raise RuntimeError(f"Sequence {seq}: annotation {ann_id} lacks tracking_id")
        anns_by_image[int(ann["image_id"])].append(ann)

    image_rows = list(coco["images"])
    image_rows.sort(key=lambda x: natural_key(x["file_name"]))
    if len(image_rows) != EXPECTED_SOURCE_IMAGE_COUNTS[seq]:
        raise RuntimeError(f"Sequence {seq}: tracked JSON has {len(image_rows)} image records; expected {EXPECTED_SOURCE_IMAGE_COUNTS[seq]}.")
    if len({int(x["id"]) for x in image_rows}) != len(image_rows):
        raise RuntimeError(f"Sequence {seq}: duplicate image IDs in tracked JSON")

    image_index = build_image_index(root)
    frame_meta_by_idx = {}
    for frame_idx, image_meta in enumerate(image_rows):
        frame_meta_by_idx[frame_idx] = image_meta

    seq_manifest = manifest_df[manifest_df.sequence == seq].sort_values("frame_idx")
    for row in seq_manifest.itertuples(index=False):
        idx = int(row.frame_idx)
        if idx not in frame_meta_by_idx: raise KeyError(f"Sequence {seq}: manifest frame {idx} outside source timeline")
        source_meta = frame_meta_by_idx[idx]
        if str(source_meta["file_name"]) != str(row.file_name):
            raise RuntimeError(f"Sequence {seq} frame {idx}: filename mismatch: manifest={row.file_name!r}, source={source_meta['file_name']!r}")
        if int(source_meta["id"]) != int(row.image_id):
            raise RuntimeError(f"Sequence {seq} frame {idx}: COCO image ID mismatch: manifest={row.image_id}, source={source_meta['id']}")
        actual_count = len(anns_by_image.get(int(source_meta["id"]), []))
        if actual_count != int(row.annotations):
            raise RuntimeError(f"Sequence {seq} frame {idx}: annotation count mismatch: manifest={row.annotations}, source={actual_count}")

    eval_rows = seq_manifest[seq_manifest.split.isin(["val", "test"])].copy()
    eval_frames = []
    for row in eval_rows.itertuples(index=False):
        meta = frame_meta_by_idx[int(row.frame_idx)]
        image_path = resolve_image_path(meta["file_name"], tracked_path, root, image_index, seq)
        eval_frames.append({
            "SEQUENCE_ID": int(seq), "FRAME_ID": int(row.frame_idx), "FRAME_KEY": f"seq{seq}_frame{int(row.frame_idx):05d}",
            "SPLIT": str(row.split), "COCO_IMAGE_ID": int(row.image_id), "FILE_NAME": str(row.file_name),
            "IMAGE_PATH": str(image_path), "IMAGE_WIDTH_META": int(meta.get("width", 0)),
            "IMAGE_HEIGHT_META": int(meta.get("height", 0)),
            "CAPTURE_TIMESTAMP_RAW": meta.get("date_captured") or meta.get("timestamp"),
            "ELAPSED_MINUTES_NOMINAL": float(int(row.frame_idx) * 15),
            "ANNOTATIONS": sorted(anns_by_image.get(int(row.image_id), []), key=lambda a: int(a["id"])),
            "CATEGORIES": categories,
        })

    identity_rows = []
    for fr in eval_frames:
        tids = [str(a["tracking_id"]) for a in fr["ANNOTATIONS"]]
        if len(tids) != len(set(tids)):
            raise RuntimeError(f"Sequence {seq} frame {fr['FRAME_ID']}: duplicate track IDs within the same frame")
        for ann in fr["ANNOTATIONS"]:
            identity_rows.append({"SEQUENCE_ID": seq, "FRAME_ID": fr["FRAME_ID"],
                                  "COCO_IMAGE_ID": fr["COCO_IMAGE_ID"],
                                  "COCO_ANNOTATION_ID": int(ann["id"]), "GT_TRACK_ID": str(ann["tracking_id"])})

    print(f"Sequence {seq}: source={source_mode}; tracked annotations={tracked_path}; images={len(image_rows)}; eval frames={len(eval_frames)}; identity rows={len(identity_rows)}")
    return {"sequence": seq, "root": Path(root), "source_mode": source_mode, "source_archive_sha256": source_archive_sha,
            "tracked_json_path": tracked_path, "tracked_json_sha256": sha256_file(tracked_path),
            "categories": categories, "anns_by_image": anns_by_image, "image_meta_by_idx": frame_meta_by_idx,
            "source_track_ids": source_track_ids, "eval_frames": eval_frames, "identity_rows": identity_rows}

SEQUENCE_DATA = {}
for seq in [1, 2, 3, 4]:
    SEQUENCE_DATA[seq] = load_sequence_bundle(seq, manifest)

IDENTITY_MAP_DF = pd.concat([pd.DataFrame(SEQUENCE_DATA[s]["identity_rows"]) for s in [1,2,3,4]], ignore_index=True)
if len(IDENTITY_MAP_DF) != 6084:
    raise RuntimeError(f"Expected exactly 6,084 evaluation source track mappings; got {len(IDENTITY_MAP_DF)}")
if IDENTITY_MAP_DF.duplicated(["SEQUENCE_ID", "FRAME_ID", "COCO_ANNOTATION_ID"]).any():
    raise RuntimeError("Duplicate sequence/frame/annotation identity mapping keys")
# The prior audit's 12/8/12/11 counts covered EVERY annotated source frame.
# This notebook's IDENTITY_MAP_DF intentionally contains only the 961 val/test frames,
# so its unique-ID counts are smaller when an identity never appears in the eval subset.
EXPECTED_FULL_SOURCE_TRACK_IDS = {1:12, 2:8, 3:12, 4:11}
EXPECTED_EVAL_SUBSET_TRACK_IDS = {1:11, 2:7, 3:12, 4:10}
actual_full_source_unique = {s: len(SEQUENCE_DATA[s]["source_track_ids"]) for s in [1,2,3,4]}
actual_eval_unique = IDENTITY_MAP_DF.groupby("SEQUENCE_ID").GT_TRACK_ID.nunique().to_dict()

if actual_full_source_unique != EXPECTED_FULL_SOURCE_TRACK_IDS:
    raise RuntimeError(
        f"Full-source tracking-ID cardinalities changed: {actual_full_source_unique}; "
        f"previous full-source audit expected {EXPECTED_FULL_SOURCE_TRACK_IDS}"
    )
if actual_eval_unique != EXPECTED_EVAL_SUBSET_TRACK_IDS:
    raise RuntimeError(
        f"Evaluation-subset tracking-ID cardinalities changed: {actual_eval_unique}; "
        f"canonical manifest audit expected {EXPECTED_EVAL_SUBSET_TRACK_IDS}"
    )

scope_rows = []
for seq in [1,2,3,4]:
    full_ids = set(SEQUENCE_DATA[seq]["source_track_ids"])
    eval_ids = set(IDENTITY_MAP_DF.loc[IDENTITY_MAP_DF.SEQUENCE_ID == seq, "GT_TRACK_ID"].astype(str))
    if not eval_ids.issubset(full_ids):
        raise RuntimeError(f"Sequence {seq}: evaluation IDs absent from full-source annotation IDs: {sorted(eval_ids-full_ids)}")
    scope_rows.append({
        "SEQUENCE_ID": seq,
        "FULL_SOURCE_UNIQUE_TRACK_IDS": len(full_ids),
        "EVAL_SUBSET_UNIQUE_TRACK_IDS": len(eval_ids),
        "SOURCE_IDS_NOT_OBSERVED_IN_EVAL": json.dumps(sorted(full_ids-eval_ids)),
        "EVAL_IDS_NOT_PRESENT_IN_SOURCE": json.dumps(sorted(eval_ids-full_ids)),
    })
SOURCE_TRACK_ID_SCOPE_AUDIT_DF = pd.DataFrame(scope_rows)
SOURCE_TRACK_ID_SCOPE_AUDIT_DF.to_csv(OUTPUT_DIR / "source_tracking_id_scope_audit.csv", index=False)

print("Evaluation annotation-to-track mapping coverage: 100% (6,084/6,084)")
print("Full-source unique IDs:", actual_full_source_unique, "| total:", sum(actual_full_source_unique.values()))
print("IDs visible in the 961-frame val/test subset:", actual_eval_unique, "| total:", sum(actual_eval_unique.values()))
print("The three identities not observed in eval remain present in the full source annotations; this is a split-coverage distinction, not a mapping failure.")
display(SOURCE_TRACK_ID_SCOPE_AUDIT_DF)
display(IDENTITY_MAP_DF.groupby("SEQUENCE_ID").agg(ANNOTATIONS=("GT_TRACK_ID", "size"), UNIQUE_TRACK_IDS=("GT_TRACK_ID", "nunique")))

Sequence 1: source=attached_extracted; tracked annotations=/kaggle/input/datasets/hnamedu/charisis-full/1st_timelapse/1st_timelapse/annotations/1st_timelapse_annotations_tracked.json; images=2084; eval frames=261; identity rows=1014
Sequence 2: source=attached_extracted; tracked annotations=/kaggle/input/datasets/hnamedu/charisis-full/2nd_timelapse/2nd_timelapse/annotations/2nd_timelapse_annotations_tracked.json; images=1175; eval frames=235; identity rows=1638
Sequence 3: source=attached_extracted; tracked annotations=/kaggle/input/datasets/hnamedu/charisis-full/3rd_timelapse/3rd_timelapse/annotations/3rd_timelapse_annotations_tracked.json; images=2133; eval frames=267; identity rows=1649
Sequence 4: source=attached_extracted; tracked annotations=/kaggle/input/datasets/hnamedu/charisis-full/4th_timelapse/4th_timelapse/annotations/4th_timelapse_annotations_tracked.json; images=990; eval frames=198; identity rows=1783
Evaluation annotation-to-track mapping coverage: 100% (6,084/6,084)
F

,SEQUENCE_ID,FULL_SOURCE_UNIQUE_TRACK_IDS,EVAL_SUBSET_UNIQUE_TRACK_IDS,SOURCE_IDS_NOT_OBSERVED_IN_EVAL,EVAL_IDS_NOT_PRESENT_IN_SOURCE
0,1,12,11,"[""8""]",[]
1,2,8,7,"[""4""]",[]
2,3,12,12,[],[]
3,4,11,10,"[""3""]",[]


,ANNOTATIONS,UNIQUE_TRACK_IDS
SEQUENCE_ID,,
1,1014,11
2,1638,7
3,1649,12
4,1783,10


In [7]:
train_manifest = manifest[(manifest.split == "train") & (manifest.sequence.isin([2,4]))].copy()
reference_rows = []
reference_frames = set()
for seq in [2,4]:
    bundle = SEQUENCE_DATA[seq]
    categories = bundle["categories"]
    reference_cat_ids = [cid for cid, name in categories.items() if name == EXPECTED_CATEGORY_REFERENCE]
    if len(reference_cat_ids) != 1:
        raise RuntimeError(f"Sequence {seq}: expected one source category named {EXPECTED_CATEGORY_REFERENCE!r}; found {reference_cat_ids} in {categories}")
    ref_cat_id = reference_cat_ids[0]
    anns_by_image = bundle["anns_by_image"]
    image_meta_by_idx = bundle["image_meta_by_idx"]
    for row in train_manifest[train_manifest.sequence == seq].itertuples(index=False):
        image_meta = image_meta_by_idx[int(row.frame_idx)]
        h, w = int(image_meta["height"]), int(image_meta["width"])
        if h <= 0 or w <= 0: raise RuntimeError("Training annotation image metadata is missing width/height")
        matched = [a for a in anns_by_image.get(int(row.image_id), []) if int(a["category_id"]) == ref_cat_id and not bool(a.get("iscrowd",0))]
        if matched: reference_frames.add((seq, int(row.frame_idx)))
        for ann in matched:
            rle = annotation_to_rle(ann, h, w)
            area = int(mask_utils.area(rle))
            if area <= 0: raise RuntimeError(f"Empty training reference annotation {ann['id']}")
            reference_rows.append({"SEQUENCE_ID": seq, "FRAME_ID": int(row.frame_idx),
                                   "COCO_ANNOTATION_ID": int(ann["id"]), "ORIGINAL_CATEGORY_NAME": EXPECTED_CATEGORY_REFERENCE,
                                   "AREA_PX": area})

reference_df = pd.DataFrame(reference_rows)
A_REF_PX2 = float(reference_df.AREA_PX.mean())
if len(reference_df) != 3365 or len(reference_frames) != 644:
    raise RuntimeError(f"Training-only reference changed: {len(reference_df)} annotations across {len(reference_frames)} frames; expected 3,365 across 644.")
if not np.isclose(A_REF_PX2, 669444.1598811293, rtol=0, atol=0.5):
    raise RuntimeError(f"Training-only reference area mismatch: {A_REF_PX2:.6f}, expected 669444.159881.")
print("A_ref (training annotations only):", A_REF_PX2, "px^2")
print("Observations:", len(reference_df), "| Frames:", len(reference_frames))
(OUTPUT_DIR / "well_defined_reference_size.json").write_text(json.dumps({
    "reference_source_category_raw": EXPECTED_CATEGORY_REFERENCE,
    "reference_paper_label": "well-defined",
    "reference_area_px2_mean": A_REF_PX2,
    "reference_area_px2_median": float(reference_df.AREA_PX.median()),
    "reference_annotation_observations": int(len(reference_df)),
    "reference_frames": int(len(reference_frames)),
    "reference_split": "train only",
    "definition": "SIZE_INDEX_PCT = AREA_PX / training mean area for source category mature_cluster * 100",
    "interpretation": "relative image-space size index, not maturity probability/classification; values above 100 are valid",
    "sampling_unit_caveat": "annotation observations, not unique physical clusters"
}, indent=2), encoding="utf-8")
reference_df.to_csv(OUTPUT_DIR / "training_reference_annotations.csv", index=False)

A_ref (training annotations only): 669444.1598811293 px^2
Observations: 3365 | Frames: 644


## Models and inference

Both models are loaded from the project checkpoints. YOLO inference retains the recorded image size, confidence floor, NMS settings, and mask export. Mask R-CNN retains the project architecture and its memory-safe postprocessing: source-coordinate boxes are resized by TorchVision, while full-resolution masks are pasted one instance at a time on the CPU. A parity check compares this operation with TorchVision postprocessing.

In [8]:
# YOLO26-L-seg: one foreground class, selected project checkpoint only.
yolo_model = YOLO(str(YOLO_MODEL_PATH))
if isinstance(yolo_model.names, dict):
    yolo_names = {int(k): str(v) for k,v in yolo_model.names.items()}
else:
    yolo_names = {i: str(v) for i,v in enumerate(yolo_model.names)}
if len(yolo_names) != 1 or 0 not in yolo_names:
    raise RuntimeError(f"Expected a single foreground class at ID 0; model exposes {yolo_names}")
print("YOLO classes:", yolo_names)


class ConvNeXtTinyFPN(nn.Module):
    """Architecture from the supplied executed Mask R-CNN baseline notebook."""
    def __init__(self):
        super().__init__()
        base = torchvision.models.convnext_tiny(weights=None).features
        self.stem = base[0]
        self.stage1 = base[1]
        self.down2 = base[2]
        self.stage2 = base[3]
        self.down3 = base[4]
        self.stage3 = base[5]
        self.down4 = base[6]
        self.stage4 = base[7]
        self.out_channels = 256
        self.fpn = FeaturePyramidNetwork(
            in_channels_list=[96, 192, 384, 768], out_channels=256,
            extra_blocks=LastLevelMaxPool())

    def forward(self, x):
        x = self.stem(x)
        f1 = self.stage1(x)
        x = self.down2(f1); f2 = self.stage2(x)
        x = self.down3(f2); f3 = self.stage3(x)
        x = self.down4(f3); f4 = self.stage4(x)
        return self.fpn({"0":f1, "1":f2, "2":f3, "3":f4})

checkpoint = torch.load(MASKRCNN_CHECKPOINT_PATH, map_location="cpu", weights_only=False)
if "model" not in checkpoint:
    raise RuntimeError("Mask R-CNN checkpoint does not contain the expected 'model' state dictionary")
state_dict = checkpoint["model"]
num_classes = int(state_dict["roi_heads.box_predictor.cls_score.weight"].shape[0])
maskrcnn_model = MaskRCNN(
    ConvNeXtTinyFPN(), num_classes=num_classes,
    box_score_thresh=MASKRCNN_SCORE_FLOOR,
    box_detections_per_img=MASKRCNN_MAX_DET,
)
maskrcnn_model.load_state_dict(state_dict, strict=True)
maskrcnn_model.roi_heads.score_thresh = MASKRCNN_SCORE_FLOOR
del state_dict, checkpoint
gc.collect()
maskrcnn_model.to(TORCH_DEVICE).eval()

class_mapping = None
if MASKRCNN_CLASS_MAP_PATH and MASKRCNN_CLASS_MAP_PATH.is_file():
    class_mapping = json.loads(MASKRCNN_CLASS_MAP_PATH.read_text(encoding="utf-8"))
print("Mask R-CNN classes incl. background:", num_classes)
print("Mask R-CNN class mapping:", class_mapping)
print("Both checkpoint hashes verified. No prediction cache has been loaded.")

YOLO classes: {0: 'mushroom_cluster'}
Mask R-CNN classes incl. background: 4
Mask R-CNN class mapping: {'label_to_name': {'1': 'immature_cluster', '2': 'mature_cluster', '3': 'overstayed_cluster'}}
Both checkpoint hashes verified. No prediction cache has been loaded.


In [9]:
def sync_cuda():
    if torch.cuda.is_available(): torch.cuda.synchronize(0)


def binary_to_rle(binary):
    encoded = mask_utils.encode(np.asfortranarray((np.asarray(binary) > 0).astype(np.uint8)))
    return rle_to_json(encoded)


def add_size_index(instance):
    instance["SIZE_INDEX_PCT"] = float(instance["AREA_PX"]) / A_REF_PX2 * 100.0
    return instance


def gt_instances_for_frame(frame):
    h, w = int(frame["IMAGE_HEIGHT"]), int(frame["IMAGE_WIDTH"])
    output = []
    anns = frame["ANNOTATIONS"]
    for local_idx, ann in enumerate(anns):
        item = annotation_to_instance(ann, h, w, local_idx, frame["CATEGORIES"])
        add_size_index(item)
        output.append(item)
    return output


def yolo_predictions_from_result(result, height, width):
    if result.boxes is None or result.masks is None: return [], 0
    scores = result.boxes.conf.detach().cpu().numpy().astype(np.float32)
    classes = result.boxes.cls.detach().cpu().numpy().astype(np.int32)
    if len(result.masks) != len(scores) or len(classes) != len(scores):
        raise RuntimeError("YOLO boxes/classes/masks length mismatch")
    output=[]; empty=0
    for raw_idx, (score, class_id) in enumerate(zip(scores, classes)):
        if float(score) < YOLO_SCORE_FLOOR: continue
        single_mask = result.masks[raw_idx]
        one = single_mask.data
        arr = one.detach().cpu().numpy() if isinstance(one, torch.Tensor) else np.asarray(one)
        if arr.ndim == 3:
            if arr.shape[0] != 1: raise RuntimeError(f"Unexpected per-instance YOLO mask shape {arr.shape}")
            arr = arr[0]
        if arr.shape != (height,width):
            raise RuntimeError(f"YOLO retina mask is {arr.shape}, expected source {(height,width)}. No silent resizing is permitted.")
        binary = (arr >= 0.5).astype(np.uint8)
        geom = tight_geometry(binary)
        if geom is None:
            empty += 1
            continue
        if int(class_id) != 0: raise RuntimeError(f"Unexpected YOLO class ID {class_id}; expected 0")
        item = {"FRAME_INSTANCE_ID": len(output), "MODEL_RAW_INSTANCE_INDEX": int(raw_idx),
                "CLASS_ID": 0, "CLASS_NAME": "mushroom_cluster", "CONFIDENCE": float(score),
                **geom, "MASK_RLE": binary_to_rle(binary)}
        add_size_index(item); output.append(item)
        del single_mask, one, arr, binary
    return output, empty


def maskrcnn_boxes_only_postprocess(result, image_shapes, original_image_sizes):
    """Keep TorchVision's box scaling but defer full-resolution mask pasting to CPU.

    The default GeneralizedRCNNTransform.postprocess expands every ROI mask into
    a full source-resolution HxW tensor on the GPU and stacks all detections.
    At the source image dimensions used here, that creates multi-GiB temporary
    allocations. This is the same memory-safe implementation already used in
    the project's canon_memory_safe.ipynb: boxes are resized with TorchVision's
    resize_boxes, while masks remain compact ROI masks until export.
    """
    for prediction, image_shape, original_image_size in zip(
        result, image_shapes, original_image_sizes
    ):
        prediction["boxes"] = resize_boxes(
            prediction["boxes"], image_shape, original_image_size
        )
    return result


# Override only the final mask-expansion step. Model weights, input transform,
# ROI inference, scores, labels, and source-coordinate boxes remain unchanged.
maskrcnn_model.transform.postprocess = maskrcnn_boxes_only_postprocess
print("Mask R-CNN postprocess: CPU per-instance TorchVision mask pasting (memory-safe mode)")


def maskrcnn_predictions_from_output(output, height, width):
    scores = output["scores"].detach().float().cpu().numpy()
    labels = output["labels"].detach().int().cpu().numpy()
    masks = output["masks"]  # compact ROI masks (N x 1 x M x M), not H x W masks
    boxes = output["boxes"]  # TorchVision-resized to original source-image coordinates
    if masks.ndim != 4 or masks.shape[1] != 1:
        raise RuntimeError(
            f"Expected Mask R-CNN ROI masks with shape N x 1 x M x M; got {tuple(masks.shape)}"
        )
    if len(boxes) != len(masks) or len(scores) != len(masks) or len(labels) != len(masks):
        raise RuntimeError("Mask R-CNN boxes, scores, labels, and masks have inconsistent counts")

    rows=[]; empty=0
    for idx, score in enumerate(scores):
        if float(score) < MASKRCNN_SCORE_FLOOR:
            continue
        # Paste one ROI mask at a time on CPU to avoid N x H x W GPU allocation.
        roi_mask_cpu = masks[idx:idx+1].detach().to("cpu")
        box_cpu = boxes[idx:idx+1].detach().to("cpu")
        pasted = paste_masks_in_image(roi_mask_cpu, box_cpu, (height, width))
        binary = (pasted[0,0].numpy() >= MASKRCNN_MASK_THRESHOLD).astype(np.uint8)
        geom = tight_geometry(binary)
        if geom is None:
            empty += 1
        else:
            item = {"FRAME_INSTANCE_ID": len(rows), "CLASS_ID": 0, "CLASS_NAME": "mushroom_cluster",
                    "ORIGINAL_CLASS_ID": int(labels[idx]), "CONFIDENCE": float(score),
                    **geom, "MASK_RLE": binary_to_rle(binary)}
            add_size_index(item); rows.append(item)
        del roi_mask_cpu, box_cpu, pasted, binary
    return rows, empty


def make_frame_record(frame, gt, predictions, model_name, checkpoint_sha, inference_ms, export_ms, empty_masks):
    return {
        "CACHE_SCHEMA": "fungicirno.fresh_inference.v1", "MODEL_NAME": model_name,
        "MODEL_CHECKPOINT_SHA256": checkpoint_sha,
        "SEQUENCE_ID": int(frame["SEQUENCE_ID"]), "FRAME_ID": int(frame["FRAME_ID"]),
        "FRAME_KEY": frame["FRAME_KEY"], "SPLIT": frame["SPLIT"],
        "COCO_IMAGE_ID": int(frame["COCO_IMAGE_ID"]), "FILE_NAME": frame["FILE_NAME"],
        "SOURCE_IMAGE_SHA256": frame.get("SOURCE_IMAGE_SHA256"),
        "IMAGE_WIDTH": int(frame["IMAGE_WIDTH"]), "IMAGE_HEIGHT": int(frame["IMAGE_HEIGHT"]),
        "CAPTURE_TIMESTAMP_RAW": frame.get("CAPTURE_TIMESTAMP_RAW"),
        "ELAPSED_MINUTES_NOMINAL": float(frame["ELAPSED_MINUTES_NOMINAL"]),
        "INFERENCE_MS": float(inference_ms), "MASK_EXPORT_MS": float(export_ms),
        "DROPPED_EMPTY_PREDICTION_MASKS": int(empty_masks),
        "PREDICTIONS": predictions, "GROUND_TRUTH_INSTANCES": gt,
    }

print("Fresh inference helpers ready. Masks are retained as full-source-resolution COCO RLE.")

Mask R-CNN postprocess: CPU per-instance TorchVision mask pasting (memory-safe mode)
Fresh inference helpers ready. Masks are retained as full-source-resolution COCO RLE.


In [10]:
def verify_maskrcnn_cpu_paste_equivalence():
    """Check CPU per-instance paste equals TorchVision's standard postprocess."""
    generator = torch.Generator(device="cpu").manual_seed(20261119)
    roi_masks = torch.rand((3, 1, 28, 28), generator=generator)
    boxes = torch.tensor([[5., 7., 35., 42.], [20., 15., 60., 70.], [0., 0., 50., 50.]])
    prediction = {
        "boxes": boxes.clone(),
        "scores": torch.tensor([0.9, 0.8, 0.7]),
        "labels": torch.tensor([1, 2, 1]),
        "masks": roi_masks.clone(),
    }
    image_shapes = [(100, 120)]
    original_sizes = [(200, 240)]

    standard_transform = GeneralizedRCNNTransform(
        min_size=100, max_size=200, image_mean=[0.0, 0.0, 0.0],
        image_std=[1.0, 1.0, 1.0]
    ).eval()
    standard = standard_transform.postprocess(
        [{key: value.clone() for key, value in prediction.items()}],
        image_shapes, original_sizes
    )[0]

    boxes_only = {key: value.clone() for key, value in prediction.items()}
    boxes_only["boxes"] = resize_boxes(
        boxes_only["boxes"], image_shapes[0], original_sizes[0]
    )
    pasted_one_by_one = torch.cat([
        paste_masks_in_image(
            boxes_only["masks"][idx:idx+1],
            boxes_only["boxes"][idx:idx+1],
            original_sizes[0]
        )
        for idx in range(len(boxes_only["masks"]))
    ], dim=0)

    if not torch.equal(standard["boxes"], boxes_only["boxes"]):
        raise RuntimeError("CPU mask-paste parity check failed: source-coordinate boxes differ")
    if not torch.equal(standard["masks"], pasted_one_by_one):
        max_abs = float((standard["masks"].float() - pasted_one_by_one.float()).abs().max())
        raise RuntimeError(f"CPU mask-paste parity check failed: mask tensors differ (max_abs={max_abs})")
    return True


MASKRCNN_CPU_PASTE_PARITY = verify_maskrcnn_cpu_paste_equivalence()
print("Mask R-CNN CPU mask-paste parity with standard TorchVision postprocess: PASS")


Mask R-CNN CPU mask-paste parity with standard TorchVision postprocess: PASS


In [11]:
YOLO_CACHE_PATH = CACHE_DIR / "fresh_yolo26_frame_predictions.jsonl.gz"
MASKRCNN_CACHE_PATH = CACHE_DIR / "fresh_maskrcnn_frame_predictions.jsonl.gz"


def inspect_completed_prediction_file(path, expected_model, expected_sha):
    """Validate a fresh prediction stream while retaining only small frame summaries."""
    if not path.is_file() or path.stat().st_size == 0:
        return None
    summaries = []
    keys = set()
    try:
        with gzip.open(path, "rt", encoding="utf-8") as handle:
            for line_number, line in enumerate(handle, 1):
                if not line.strip():
                    continue
                row = json.loads(line)
                if row.get("CACHE_SCHEMA") != "fungicirno.fresh_inference.v1":
                    return None
                if row.get("MODEL_NAME") != expected_model or row.get("MODEL_CHECKPOINT_SHA256") != expected_sha:
                    return None
                key = row.get("FRAME_KEY")
                if not key or key in keys or not row.get("SOURCE_IMAGE_SHA256"):
                    return None
                if "PREDICTIONS" not in row or "GROUND_TRUTH_INSTANCES" not in row:
                    return None
                keys.add(key)
                summaries.append({
                    "SEQUENCE_ID": int(row["SEQUENCE_ID"]), "FRAME_ID": int(row["FRAME_ID"]),
                    "FRAME_KEY": key, "SPLIT": row["SPLIT"], "COCO_IMAGE_ID": row["COCO_IMAGE_ID"],
                    "FILE_NAME": row["FILE_NAME"], "SOURCE_IMAGE_SHA256": row["SOURCE_IMAGE_SHA256"],
                    "IMAGE_WIDTH": row["IMAGE_WIDTH"], "IMAGE_HEIGHT": row["IMAGE_HEIGHT"],
                    "CAPTURE_TIMESTAMP_RAW": row.get("CAPTURE_TIMESTAMP_RAW"),
                    "INFERENCE_MS": row["INFERENCE_MS"], "MASK_EXPORT_MS": row["MASK_EXPORT_MS"],
                    "PREDICTION_COUNT": len(row["PREDICTIONS"]),
                    "GT_COUNT": len(row["GROUND_TRUTH_INSTANCES"]),
                    "GT_IDENTITIES": ([
                        {"GT_FRAME_INSTANCE_ID": gt["GT_FRAME_INSTANCE_ID"],
                         "COCO_ANNOTATION_ID": gt["COCO_ANNOTATION_ID"], "GT_TRACK_ID": gt["GT_TRACK_ID"]}
                        for gt in row["GROUND_TRUTH_INSTANCES"]
                    ] if expected_model == "YOLO26-L-seg" else None)
                })
    except Exception as exc:
        print(f"Existing prediction cache is incomplete or unreadable ({path.name}): {exc}")
        return None
    return summaries


expected_keys = {
    frame["FRAME_KEY"]
    for seq in [1, 2, 3, 4]
    for frame in SEQUENCE_DATA[seq]["eval_frames"]
}
existing_yolo = inspect_completed_prediction_file(
    YOLO_CACHE_PATH, "YOLO26-L-seg", YOLO_SHA256
) if REUSE_COMPLETED_FRESH_INFERENCE_IF_VALID else None
existing_mask = inspect_completed_prediction_file(
    MASKRCNN_CACHE_PATH, "Mask R-CNN + ConvNeXt-Tiny-FPN", MASKRCNN_SHA256
) if REUSE_COMPLETED_FRESH_INFERENCE_IF_VALID else None

cache_pair_valid = (
    existing_yolo is not None and existing_mask is not None
    and len(existing_yolo) == 961 and len(existing_mask) == 961
    and {r["FRAME_KEY"] for r in existing_yolo} == expected_keys
    and {r["FRAME_KEY"] for r in existing_mask} == expected_keys
)

runtime_rows = []
identity_rows = []
frame_audit_rows = []
INFERENCE_CACHE_REUSED = bool(cache_pair_valid)

if INFERENCE_CACHE_REUSED:
    print("Reusing 961 + 961 completed fresh-inference records. No model forward pass will be repeated.")
    mask_count_by_key = {r["FRAME_KEY"]: r["PREDICTION_COUNT"] for r in existing_mask}
    for row in existing_yolo:
        seq, frame_id, key = row["SEQUENCE_ID"], row["FRAME_ID"], row["FRAME_KEY"]
        frame_audit_rows.append({
            "SEQUENCE_ID": seq, "FRAME_ID": frame_id, "FRAME_KEY": key,
            "SPLIT": row["SPLIT"], "COCO_IMAGE_ID": row["COCO_IMAGE_ID"],
            "FILE_NAME": row["FILE_NAME"], "SOURCE_IMAGE_SHA256": row["SOURCE_IMAGE_SHA256"],
            "IMAGE_WIDTH": row["IMAGE_WIDTH"], "IMAGE_HEIGHT": row["IMAGE_HEIGHT"],
            "GT_COUNT": row["GT_COUNT"], "YOLO_COUNT": row["PREDICTION_COUNT"],
            "MASKRCNN_COUNT": mask_count_by_key[key], "CAPTURE_TIMESTAMP_RAW": row.get("CAPTURE_TIMESTAMP_RAW")
        })
        for gt in row["GT_IDENTITIES"]:
            identity_rows.append({
                "SEQUENCE_ID": seq, "FRAME_ID": frame_id,
                "GT_FRAME_INSTANCE_ID": gt["GT_FRAME_INSTANCE_ID"],
                "COCO_ANNOTATION_ID": gt["COCO_ANNOTATION_ID"], "GT_TRACK_ID": gt["GT_TRACK_ID"]
            })
        runtime_rows.append({
            "MODEL": "YOLO26-L-seg", "SEQUENCE_ID": seq, "FRAME_ID": frame_id, "FRAME_KEY": key,
            "INFERENCE_MS": row["INFERENCE_MS"], "MASK_EXPORT_MS": row["MASK_EXPORT_MS"],
            "PREDICTION_COUNT": row["PREDICTION_COUNT"], "HOST_RSS_MB": np.nan
        })
    for row in existing_mask:
        runtime_rows.append({
            "MODEL": "Mask R-CNN + ConvNeXt-Tiny-FPN", "SEQUENCE_ID": row["SEQUENCE_ID"],
            "FRAME_ID": row["FRAME_ID"], "FRAME_KEY": row["FRAME_KEY"],
            "INFERENCE_MS": row["INFERENCE_MS"], "MASK_EXPORT_MS": row["MASK_EXPORT_MS"],
            "PREDICTION_COUNT": row["PREDICTION_COUNT"], "HOST_RSS_MB": np.nan
        })
    frame_audit_df = pd.DataFrame(frame_audit_rows)
    identity_df = pd.DataFrame(identity_rows)
    RUNTIME_DETAILS_AVAILABLE = False
    print("Prediction records validated against model hashes, source-image hashes, frame keys, and frame counts.")
else:
    print("No complete matching fresh-inference cache pair found; running both models on all 961 frames.")
    for cache_path in [YOLO_CACHE_PATH, MASKRCNN_CACHE_PATH]:
        cache_path.unlink(missing_ok=True)
    RUNTIME_DETAILS_AVAILABLE = True
    process = psutil.Process(os.getpid())
    torch.cuda.reset_peak_memory_stats(0)

    runtime_rows=[]
    identity_rows=[]
    frame_audit_rows=[]
    process = psutil.Process(os.getpid())
    torch.cuda.reset_peak_memory_stats(0)


    def warm_up_models(eval_frames):
        warm = eval_frames[:min(WARMUP_IMAGES_PER_SEQUENCE, len(eval_frames))]
        for fr in warm:
            img = cv2.imread(fr["IMAGE_PATH"], cv2.IMREAD_COLOR)
            if img is None: raise RuntimeError(f"Could not read warm-up image {fr['IMAGE_PATH']}")
            # Match the supplied YOLO26 inference notebook exactly: one source image,
            # imgsz=1024, conf=0.001, iou=0.70, nms=None, max_det=1000, retina_masks=True.
            sync_cuda()
            yolo_warm_result = yolo_model.predict(
                source=img, imgsz=YOLO_IMAGE_SIZE, conf=YOLO_SCORE_FLOOR,
                iou=YOLO_NMS_IOU, nms=YOLO_NMS_MODE, max_det=YOLO_MAX_DET, device=DEVICE,
                retina_masks=YOLO_RETINA_MASKS, save=False, verbose=False
            )[0]
            sync_cuda()
            del yolo_warm_result
            gc.collect()

            # Warm up Mask R-CNN only after releasing YOLO's full-resolution result.
            rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            tensor = TVF.to_tensor(rgb).to(TORCH_DEVICE)
            with torch.inference_mode():
                mask_warm_output = maskrcnn_model([tensor])[0]
            sync_cuda()
            del mask_warm_output, tensor, rgb, img
            gc.collect()
            torch.cuda.empty_cache()
        gc.collect(); torch.cuda.empty_cache()


    with gzip.open(YOLO_CACHE_PATH, "wt", encoding="utf-8") as yolo_out, gzip.open(MASKRCNN_CACHE_PATH, "wt", encoding="utf-8") as mask_out:
        for seq in [1,2,3,4]:
            frames = SEQUENCE_DATA[seq]["eval_frames"]
            if not frames: raise RuntimeError(f"Sequence {seq} has no evaluation frames")
            print(f"\n=== Fresh inference: sequence {seq} ({len(frames)} frames) ===")
            warm_up_models(frames)
            for position, frame in enumerate(tqdm(frames, desc=f"Sequence {seq}")):
                image = cv2.imread(frame["IMAGE_PATH"], cv2.IMREAD_COLOR)
                if image is None: raise RuntimeError(f"Could not read source image: {frame['IMAGE_PATH']}")
                height, width = image.shape[:2]
                frame["SOURCE_IMAGE_SHA256"] = sha256_file(frame["IMAGE_PATH"])
                if frame["IMAGE_WIDTH_META"] and frame["IMAGE_HEIGHT_META"]:
                    if (width,height) != (frame["IMAGE_WIDTH_META"],frame["IMAGE_HEIGHT_META"]):
                        raise RuntimeError(f"Source image dimension mismatch: {frame['FRAME_KEY']} actual={(width,height)} COCO={(frame['IMAGE_WIDTH_META'],frame['IMAGE_HEIGHT_META'])}")
                frame["IMAGE_WIDTH"], frame["IMAGE_HEIGHT"] = width, height
                gt = gt_instances_for_frame(frame)
                identity_rows.extend([{"SEQUENCE_ID":seq,"FRAME_ID":frame["FRAME_ID"],
                                       "GT_FRAME_INSTANCE_ID":i,"COCO_ANNOTATION_ID":g["COCO_ANNOTATION_ID"],
                                       "GT_TRACK_ID":g["GT_TRACK_ID"]} for i,g in enumerate(gt)])
                if len(gt) != len(frame["ANNOTATIONS"]): raise RuntimeError("GT instance materialization count mismatch")

                # Preserve a paper-sized source thumbnail for post-hoc visual selection.
                if SAVE_SOURCE_THUMBNAILS:
                    scale = min(THUMBNAIL_WIDTH / width, 1.0)
                    thumb = cv2.resize(image, (int(round(width*scale)), int(round(height*scale))), interpolation=cv2.INTER_AREA)
                    thumb_path = THUMBNAIL_DIR / f"seq{seq}_frame{int(frame['FRAME_ID']):05d}.jpg"
                    if not cv2.imwrite(str(thumb_path), thumb, [int(cv2.IMWRITE_JPEG_QUALITY), THUMBNAIL_JPEG_QUALITY]):
                        raise RuntimeError(f"Could not write thumbnail {thumb_path}")

                # YOLO model forward + Ultralytics post-processing timing; synchronized on both ends.
                sync_cuda(); t0 = time.perf_counter()
                yolo_result = yolo_model.predict(source=image, imgsz=YOLO_IMAGE_SIZE, conf=YOLO_SCORE_FLOOR,
                                                 iou=YOLO_NMS_IOU, nms=YOLO_NMS_MODE, max_det=YOLO_MAX_DET, device=DEVICE,
                                                 retina_masks=YOLO_RETINA_MASKS, save=False, verbose=False)[0]
                sync_cuda(); yolo_infer_ms = (time.perf_counter()-t0)*1000.0
                t0 = time.perf_counter()
                yolo_preds, yolo_empty = yolo_predictions_from_result(yolo_result, height, width)
                yolo_export_ms = (time.perf_counter()-t0)*1000.0
                yolo_record = make_frame_record(frame, gt, yolo_preds, "YOLO26-L-seg", YOLO_SHA256,
                                                yolo_infer_ms, yolo_export_ms, yolo_empty)
                yolo_out.write(json.dumps(yolo_record, ensure_ascii=False, separators=(",", ":")) + "\n")
                runtime_rows.append({"MODEL":"YOLO26-L-seg","SEQUENCE_ID":seq,"FRAME_ID":frame["FRAME_ID"],
                                     "FRAME_KEY":frame["FRAME_KEY"],"INFERENCE_MS":yolo_infer_ms,
                                     "MASK_EXPORT_MS":yolo_export_ms,"PREDICTION_COUNT":len(yolo_preds),
                                     "HOST_RSS_MB":process.memory_info().rss/2**20})
                # Release full-resolution YOLO mask tensors before the second model's forward pass.
                del yolo_result
                gc.collect()

                # Mask R-CNN: network/ROI inference is unchanged. Only full-resolution mask pasting
                # is moved to CPU per instance using TorchVision paste_masks_in_image (see cell 17).
                rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
                tensor = TVF.to_tensor(rgb).to(TORCH_DEVICE)
                sync_cuda(); t0 = time.perf_counter()
                with torch.inference_mode(): mask_output = maskrcnn_model([tensor])[0]
                sync_cuda(); mask_infer_ms = (time.perf_counter()-t0)*1000.0
                t0 = time.perf_counter()
                mask_preds, mask_empty = maskrcnn_predictions_from_output(mask_output, height, width)
                mask_export_ms = (time.perf_counter()-t0)*1000.0
                mask_record = make_frame_record(frame, gt, mask_preds, "Mask R-CNN + ConvNeXt-Tiny-FPN",
                                                MASKRCNN_SHA256, mask_infer_ms, mask_export_ms, mask_empty)
                mask_out.write(json.dumps(mask_record, ensure_ascii=False, separators=(",", ":")) + "\n")
                runtime_rows.append({"MODEL":"Mask R-CNN + ConvNeXt-Tiny-FPN","SEQUENCE_ID":seq,"FRAME_ID":frame["FRAME_ID"],
                                     "FRAME_KEY":frame["FRAME_KEY"],"INFERENCE_MS":mask_infer_ms,
                                     "MASK_EXPORT_MS":mask_export_ms,"PREDICTION_COUNT":len(mask_preds),
                                     "HOST_RSS_MB":process.memory_info().rss/2**20})

                frame_audit_rows.append({"SEQUENCE_ID":seq,"FRAME_ID":frame["FRAME_ID"],"FRAME_KEY":frame["FRAME_KEY"],
                                         "SPLIT":frame["SPLIT"],"COCO_IMAGE_ID":frame["COCO_IMAGE_ID"],
                                         "FILE_NAME":frame["FILE_NAME"],"SOURCE_IMAGE_SHA256":frame["SOURCE_IMAGE_SHA256"],
                                         "IMAGE_WIDTH":width,"IMAGE_HEIGHT":height,
                                         "GT_COUNT":len(gt),"YOLO_COUNT":len(yolo_preds),"MASKRCNN_COUNT":len(mask_preds),
                                         "CAPTURE_TIMESTAMP_RAW":frame.get("CAPTURE_TIMESTAMP_RAW")})
                # All names below are assigned on every successful pass through the loop.
                # yolo_result was already released immediately after YOLO export above.
                del yolo_preds, mask_output, mask_preds, tensor, rgb, image, gt, yolo_record, mask_record
                if (position+1) % 20 == 0:
                    gc.collect(); torch.cuda.empty_cache()

            # Release only working-directory extractions; attached /kaggle/input data is read-only.
            if SEQUENCE_DATA[seq]["source_mode"] in {"attached_archive", "google_drive"}:
                shutil.rmtree(SEQUENCE_DATA[seq]["root"], ignore_errors=True)
            gc.collect(); torch.cuda.empty_cache()

    print("Fresh YOLO records:", YOLO_CACHE_PATH, "| bytes:", YOLO_CACHE_PATH.stat().st_size)
    print("Fresh Mask R-CNN records:", MASKRCNN_CACHE_PATH, "| bytes:", MASKRCNN_CACHE_PATH.stat().st_size)
    runtime_df = pd.DataFrame(runtime_rows)
    runtime_df.to_csv(OUTPUT_DIR / "fresh_runtime_per_frame.csv", index=False)
    pd.DataFrame(frame_audit_rows).to_csv(OUTPUT_DIR / "fresh_frame_integrity.csv", index=False)
    pd.DataFrame(identity_rows).to_csv(OUTPUT_DIR / "fresh_identity_observations.csv", index=False)

# Keep the post-inference cells independent of whether this cell ran inference or resumed caches.
runtime_df = pd.DataFrame(runtime_rows)
if INFERENCE_CACHE_REUSED:
    runtime_df.to_csv(OUTPUT_DIR / "fresh_runtime_per_frame.csv", index=False)
    pd.DataFrame(frame_audit_rows).to_csv(OUTPUT_DIR / "fresh_frame_integrity.csv", index=False)
    pd.DataFrame(identity_rows).to_csv(OUTPUT_DIR / "fresh_identity_observations.csv", index=False)


No complete matching fresh-inference cache pair found; running both models on all 961 frames.

=== Fresh inference: sequence 1 (261 frames) ===


Sequence 1:   0%|          | 0/261 [00:00<?, ?it/s]


=== Fresh inference: sequence 2 (235 frames) ===


Sequence 2:   0%|          | 0/235 [00:00<?, ?it/s]


=== Fresh inference: sequence 3 (267 frames) ===


Sequence 3:   0%|          | 0/267 [00:00<?, ?it/s]


=== Fresh inference: sequence 4 (198 frames) ===


Sequence 4:   0%|          | 0/198 [00:00<?, ?it/s]

Fresh YOLO records: /kaggle/working/FungiCirno_Independent_Evidence/evidence/fresh_prediction_records/fresh_yolo26_frame_predictions.jsonl.gz | bytes: 21096619
Fresh Mask R-CNN records: /kaggle/working/FungiCirno_Independent_Evidence/evidence/fresh_prediction_records/fresh_maskrcnn_frame_predictions.jsonl.gz | bytes: 40876119


## Segmentation and geometric evaluation

Fresh prediction records are reloaded from this run and checked before evaluation. Both models are assessed under the same mask-matching protocol. COCO mask AP/AR, thresholded precision and recall, and matched-instance area, width, and height errors are exported separately. Validation-selected operating thresholds are frozen for test evaluation.

In [12]:
def load_jsonl_gzip(path):
    with gzip.open(path, "rt", encoding="utf-8") as handle:
        rows = [json.loads(line) for line in handle if line.strip()]
    rows.sort(key=lambda r:(int(r["SEQUENCE_ID"]),int(r["FRAME_ID"])))
    return rows

YOLO_RECORDS = load_jsonl_gzip(YOLO_CACHE_PATH)
MASKRCNN_RECORDS = load_jsonl_gzip(MASKRCNN_CACHE_PATH)
if len(YOLO_RECORDS) != 961 or len(MASKRCNN_RECORDS) != 961:
    raise RuntimeError(f"Fresh frame count mismatch: YOLO={len(YOLO_RECORDS)}, Mask R-CNN={len(MASKRCNN_RECORDS)}")

expected_frame_keys = set(eval_manifest.apply(lambda r:f"seq{int(r.sequence)}_frame{int(r.frame_idx):05d}", axis=1))
expected_split_by_key = {
    f"seq{int(r.sequence)}_frame{int(r.frame_idx):05d}": str(r.split)
    for r in eval_manifest.itertuples(index=False)
}
for name, records in [("YOLO",YOLO_RECORDS),("MaskRCNN",MASKRCNN_RECORDS)]:
    keys = {r["FRAME_KEY"] for r in records}
    if keys != expected_frame_keys: raise RuntimeError(f"{name} frame keys differ from the canonical manifest: missing={len(expected_frame_keys-keys)}, extra={len(keys-expected_frame_keys)}")
    for rec in records:
        if rec.get("SPLIT") != expected_split_by_key.get(rec["FRAME_KEY"]):
            raise RuntimeError(f"{name} split label disagrees with canonical manifest for {rec['FRAME_KEY']}: {rec.get('SPLIT')!r}")
        if rec.get("MODEL_CHECKPOINT_SHA256") != (YOLO_SHA256 if name == "YOLO" else MASKRCNN_SHA256):
            raise RuntimeError(f"{name} prediction record has an unexpected checkpoint digest for {rec['FRAME_KEY']}")
        for group in [rec["GROUND_TRUTH_INSTANCES"],rec["PREDICTIONS"]]:
            for obj in group:
                rle = {"size":obj["MASK_RLE"]["size"],"counts":obj["MASK_RLE"]["counts"].encode("ascii")}
                area = int(mask_utils.area(rle))
                if area != int(obj["AREA_PX"]): raise RuntimeError(f"RLE/AREA mismatch in {name} {rec['FRAME_KEY']} instance={obj.get('FRAME_INSTANCE_ID')}: {area}!={obj['AREA_PX']}")
                if list(map(int,mask_utils.toBbox(rle))) != list(map(int,obj["BBOX_XYWH"])):
                    raise RuntimeError(f"RLE/BBOX mismatch in {name} {rec['FRAME_KEY']}")

IDENTITY_MAP_DF = pd.DataFrame(identity_rows)
if len(IDENTITY_MAP_DF) != 6084: raise RuntimeError(f"Fresh source identity map expected 6084 rows, got {len(IDENTITY_MAP_DF)}")
if IDENTITY_MAP_DF.duplicated(["SEQUENCE_ID","FRAME_ID","GT_FRAME_INSTANCE_ID"]).any(): raise RuntimeError("Duplicate GT frame-instance identity keys")
if IDENTITY_MAP_DF.GT_TRACK_ID.isna().any(): raise RuntimeError("Null GT track IDs found")

pd.DataFrame(frame_audit_rows).to_csv(OUTPUT_DIR/"fresh_frame_integrity.csv",index=False)
IDENTITY_MAP_DF.to_csv(OUTPUT_DIR/"fresh_source_track_id_lookup.csv",index=False)
runtime_df = pd.DataFrame(runtime_rows)
runtime_df.to_csv(OUTPUT_DIR/"fresh_runtime_per_frame.csv",index=False)

# Cross-check the run ledger independently of prediction-record loading.
frame_integrity_df = pd.DataFrame(frame_audit_rows)
if len(frame_integrity_df) != 961 or frame_integrity_df.FRAME_KEY.nunique() != 961:
    raise RuntimeError(f"Expected 961 unique source-image integrity rows; got {len(frame_integrity_df)} rows / {frame_integrity_df.FRAME_KEY.nunique()} unique frames")
if not frame_integrity_df.SOURCE_IMAGE_SHA256.astype(str).str.fullmatch(r"[0-9a-f]{64}").all():
    raise RuntimeError("At least one evaluated source image is missing a valid SHA-256 digest")
if len(runtime_df) != 1922 or runtime_df.groupby("MODEL").FRAME_KEY.nunique().to_dict() != {
    "YOLO26-L-seg": 961, "Mask R-CNN + ConvNeXt-Tiny-FPN": 961
}:
    raise RuntimeError(f"Runtime ledger is incomplete or duplicated: rows={len(runtime_df)}, by_model={runtime_df.groupby('MODEL').FRAME_KEY.nunique().to_dict()}")
source_hashes = frame_integrity_df.set_index("FRAME_KEY").SOURCE_IMAGE_SHA256.to_dict()
for name, records in [("YOLO", YOLO_RECORDS), ("MaskRCNN", MASKRCNN_RECORDS)]:
    for rec in records:
        if rec.get("SOURCE_IMAGE_SHA256") != source_hashes.get(rec["FRAME_KEY"]):
            raise RuntimeError(f"{name} record source-image hash mismatch: {rec['FRAME_KEY']}")
RLE_GEOMETRY_CHECKS_PASSED = True  # Reached only after every prediction and GT RLE assertion above passed.
print("PASS: 961/961 frames per model; all fresh masks pass RLE area and tight-box parity.")
print("PASS: 6,084 source GT annotations mapped to persistent source tracking IDs.")

PASS: 961/961 frames per model; all fresh masks pass RLE area and tight-box parity.
PASS: 6,084 source GT annotations mapped to persistent source tracking IDs.


In [13]:
MASK_MATCH_IOU = 0.50

def coco_rle(instance):
    counts = instance["MASK_RLE"]["counts"]
    if isinstance(counts,str): counts=counts.encode("ascii")
    return {"size":[int(x) for x in instance["MASK_RLE"]["size"]],"counts":counts}


def frame_iou_matrix(predictions, ground_truth):
    if not predictions or not ground_truth:
        return np.zeros((len(predictions), len(ground_truth)), dtype=np.float64)
    dt = [coco_rle(x) for x in predictions]
    gt = [coco_rle(x) for x in ground_truth]
    # Preserve the source COCO crowd semantics. For non-crowd annotations this
    # is numerically identical to the previous all-zero iscrowd list.
    iscrowd = [int(bool(item.get("IS_CROWD", False))) for item in ground_truth]
    return mask_utils.iou(dt, gt, iscrowd).astype(np.float64)


def hungarian_min_cost(cost):
    a=np.asarray(cost,dtype=np.float64)
    if a.ndim!=2: raise ValueError("cost must be 2D")
    n,m=a.shape
    if n==0 or m==0: return np.array([],dtype=int),np.array([],dtype=int)
    if n>m:
        c,r=hungarian_min_cost(a.T); order=np.argsort(r); return r[order],c[order]
    if not np.isfinite(a).all(): raise ValueError("Non-finite assignment cost")
    u=np.zeros(n+1); v=np.zeros(m+1); p=np.zeros(m+1,dtype=np.int64); way=np.zeros(m+1,dtype=np.int64)
    for i in range(1,n+1):
        p[0]=i; j0=0; minv=np.full(m+1,np.inf); used=np.zeros(m+1,dtype=bool)
        while True:
            used[j0]=True; i0=p[j0]; delta=np.inf; j1=0
            for j in range(1,m+1):
                if not used[j]:
                    cur=a[i0-1,j-1]-u[i0]-v[j]
                    if cur<minv[j]: minv[j]=cur; way[j]=j0
                    if minv[j]<delta: delta=minv[j]; j1=j
            if not np.isfinite(delta): raise RuntimeError("No augmenting path")
            for j in range(m+1):
                if used[j]: u[p[j]]+=delta; v[j]-=delta
                else: minv[j]-=delta
            j0=j1
            if p[j0]==0: break
        while True:
            j1=way[j0]; p[j0]=p[j1]; j0=j1
            if j0==0: break
    row_to_col=np.full(n,-1,dtype=int)
    for j in range(1,m+1):
        if p[j]!=0: row_to_col[p[j]-1]=j-1
    rows=np.arange(n); keep=row_to_col>=0
    return rows[keep],row_to_col[keep]


def max_cardinality_iou_matches(iou, min_iou=MASK_MATCH_IOU):
    values=np.asarray(iou,dtype=np.float64); n,m=values.shape
    if n==0 or m==0: return []
    valid=values>=float(min_iou); bonus=float(min(n,m)+1)
    weights=np.zeros((n,m+n),dtype=np.float64)
    weights[:,:m]=np.where(valid,bonus+values,0.0)
    rows,cols=hungarian_min_cost(-weights)
    return [(int(r),int(c),float(values[r,c])) for r,c in zip(rows,cols) if c<m and valid[r,c]]


def prepare_records(records):
    out=[]
    for frame in tqdm(records,desc=f"Prepare {records[0]['MODEL_NAME'] if records else 'empty'}"):
        preds=frame["PREDICTIONS"]; gts=frame["GROUND_TRUTH_INSTANCES"]
        out.append({"frame":frame,"preds":preds,"gts":gts,"iou":frame_iou_matrix(preds,gts)})
    return out


def select_confidence_threshold(prepared):
    states=[]; events=[]; total_gt=0
    for fi,rec in enumerate(prepared):
        n_pred,n_gt=rec["iou"].shape; total_gt+=n_gt
        neighbors=[]
        for pi in range(n_pred):
            adj=np.flatnonzero(rec["iou"][pi]>=MASK_MATCH_IOU).tolist()
            adj.sort(key=lambda gj:float(rec["iou"][pi,gj]),reverse=True)
            neighbors.append(adj); events.append((float(rec["preds"][pi]["CONFIDENCE"]),fi,pi))
        states.append({"neighbors":neighbors,"gt_to_pred":[-1]*n_gt})
    events.sort(key=lambda x:(-x[0],x[1],x[2]))
    rows=[]; active=tp=0; k=0
    while k<len(events):
        score=events[k][0]; end=k
        while end<len(events) and events[end][0]==score:
            _,fi,pi=events[end]; state=states[fi]; seen=set()
            def augment(pred_idx):
                for gt_idx in state["neighbors"][pred_idx]:
                    if gt_idx in seen: continue
                    seen.add(gt_idx); prior=state["gt_to_pred"][gt_idx]
                    if prior<0 or augment(prior): state["gt_to_pred"][gt_idx]=pred_idx; return True
                return False
            if augment(pi): tp+=1
            active+=1; end+=1
        fp=active-tp; fn=total_gt-tp
        precision=tp/(tp+fp) if tp+fp else 0.0; recall=tp/(tp+fn) if tp+fn else 0.0
        f1=2*precision*recall/(precision+recall) if precision+recall else 0.0
        rows.append({"confidence_threshold":score,"TP":tp,"FP":fp,"FN":fn,"precision":precision,"recall":recall,"F1":f1,
                     "predictions_after_threshold":active,"ground_truth_instances":total_gt})
        k=end
    sweep=pd.DataFrame(rows)
    if sweep.empty: raise RuntimeError("No prediction candidates for validation threshold selection")
    best=float(sweep.F1.max()); tied=sweep[np.isclose(sweep.F1,best,atol=1e-12)]
    selected=tied.sort_values(["precision","confidence_threshold"],ascending=[False,False]).iloc[0]
    return float(selected.confidence_threshold),sweep,selected.to_dict()


def evaluate_at_threshold(prepared, threshold):
    tp=fp=fn=0; frame_rows=[]; pair_rows=[]
    for rec in prepared:
        fr=rec["frame"]; preds=rec["preds"]; gts=rec["gts"]; iou=rec["iou"]
        keep=[i for i,p in enumerate(preds) if float(p["CONFIDENCE"])>=threshold]
        sub=iou[keep,:] if keep else np.zeros((0,len(gts)),dtype=np.float64)
        pairs=max_cardinality_iou_matches(sub,MASK_MATCH_IOU)
        usedp={p for p,_,_ in pairs}; usedg={g for _,g,_ in pairs}
        ft=len(pairs); ffp=len(keep)-ft; ffn=len(gts)-ft
        tp+=ft; fp+=ffp; fn+=ffn
        frame_rows.append({"MODEL":fr["MODEL_NAME"],"SPLIT":fr["SPLIT"],"SEQUENCE_ID":fr["SEQUENCE_ID"],"FRAME_ID":fr["FRAME_ID"],
                           "FRAME_KEY":fr["FRAME_KEY"],"TP":ft,"FP":ffp,"FN":ffn})
        for pi,gi,overlap in pairs:
            pred=preds[keep[pi]]; gt=gts[gi]
            row={"MODEL":fr["MODEL_NAME"],"SPLIT":fr["SPLIT"],"SEQUENCE_ID":fr["SEQUENCE_ID"],"FRAME_ID":fr["FRAME_ID"],
                 "FRAME_KEY":fr["FRAME_KEY"],"FRAME_INSTANCE_ID":int(pred["FRAME_INSTANCE_ID"]),
                 "GT_FRAME_INSTANCE_ID":int(gt["GT_FRAME_INSTANCE_ID"]),"COCO_ANNOTATION_ID":int(gt["COCO_ANNOTATION_ID"]),
                 "GT_TRACK_ID":str(gt["GT_TRACK_ID"]),"CONFIDENCE":float(pred["CONFIDENCE"]),"MASK_IOU":float(overlap)}
            for field in ["AREA_PX","WIDTH_PX","HEIGHT_PX"]:
                pv=float(pred[field]); gv=float(gt[field])
                row.update({f"PRED_{field}":pv,f"GT_{field}":gv,f"{field}_ABS_ERROR":abs(pv-gv),
                            f"{field}_SIGNED_ERROR":pv-gv,f"{field}_REL_ABS_ERROR":abs(pv-gv)/gv if gv>0 else np.nan})
            pair_rows.append(row)
    precision=tp/(tp+fp) if tp+fp else 0.0; recall=tp/(tp+fn) if tp+fn else 0.0
    f1=2*precision*recall/(precision+recall) if precision+recall else 0.0
    summary={"TP":tp,"FP":fp,"FN":fn,"precision":precision,"recall":recall,"F1":f1,"confidence_threshold":float(threshold)}
    return summary,pd.DataFrame(frame_rows),pd.DataFrame(pair_rows)


def geometry_summary(pairs):
    rows=[]
    for field in ["AREA_PX","WIDTH_PX","HEIGHT_PX"]:
        if pairs.empty: rows.append({"MEASUREMENT":field,"N":0}); continue
        ae=pairs[f"{field}_ABS_ERROR"].to_numpy(float); se=pairs[f"{field}_SIGNED_ERROR"].to_numpy(float)
        re=pairs[f"{field}_REL_ABS_ERROR"].dropna().to_numpy(float)
        rows.append({"MEASUREMENT":field,"N":int(len(ae)),"MAE":float(np.mean(ae)),"RMSE":float(np.sqrt(np.mean(ae**2))),
                     "MEAN_RELATIVE_ABS_ERROR":float(np.mean(re)),"MEDIAN_RELATIVE_ABS_ERROR":float(np.median(re)),"SIGNED_BIAS":float(np.mean(se))})
    return pd.DataFrame(rows)


def coco_mask_ap(records):
    images=[]; annotations=[]; detections=[]; ann_id=1
    for image_idx,rec in enumerate(records,start=1):
        fr=rec["frame"]; images.append({"id":image_idx,"file_name":fr["FILE_NAME"],"height":fr["IMAGE_HEIGHT"],"width":fr["IMAGE_WIDTH"]})
        for gt in rec["gts"]:
            annotations.append({"id":ann_id,"image_id":image_idx,"category_id":1,"segmentation":coco_rle(gt),
                                "area":int(gt["AREA_PX"]),"bbox":[float(x) for x in gt["BBOX_XYWH"]],"iscrowd":int(gt.get("IS_CROWD",False))})
            ann_id+=1
        for pred in rec["preds"]:
            detections.append({"image_id":image_idx,"category_id":1,"segmentation":coco_rle(pred),"score":float(pred["CONFIDENCE"])})
    coco_gt=COCO(); coco_gt.dataset={"images":images,"categories":[{"id":1,"name":"mushroom_cluster","supercategory":"cluster"}],"annotations":annotations}; coco_gt.createIndex()
    if not detections: return {"AP":0.0,"AP50":0.0,"AP75":0.0,"AP50_95":0.0}
    coco_dt=coco_gt.loadRes(detections)
    ev=COCOeval(coco_gt,coco_dt,iouType="segm"); ev.params.catIds=[1]; ev.params.imgIds=[x["id"] for x in images]
    ev.params.maxDets=[1,10,AP_MAX_DETS]
    ev.evaluate(); ev.accumulate(); ev.summarize()
    return {"AP":float(ev.stats[0]),"AP50":float(ev.stats[1]),"AP75":float(ev.stats[2]),"AP50_95":float(ev.stats[0]),
            "AR100":float(ev.stats[8]),"AP_MAX_DETS":AP_MAX_DETS,"PROTOCOL":"pycocotools COCOeval segm; one class; full-source-resolution RLE"}


def evaluate_model(records):
    val=[r for r in records if r["SPLIT"]=="val"]; test=[r for r in records if r["SPLIT"]=="test"]
    val_p=prepare_records(val); test_p=prepare_records(test)
    threshold,sweep,selection=select_confidence_threshold(val_p)
    val_summary,val_frames,val_pairs=evaluate_at_threshold(val_p,threshold)
    test_summary,test_frames,test_pairs=evaluate_at_threshold(test_p,threshold)
    result={"model":records[0]["MODEL_NAME"],"threshold":threshold,"selection":selection,"sweep":sweep,
            "val_summary":val_summary,"test_summary":test_summary,"val_frames":val_frames,"test_frames":test_frames,
            "val_pairs":val_pairs,"test_pairs":test_pairs,"val_geometry":geometry_summary(val_pairs),
            "test_geometry":geometry_summary(test_pairs),"val_ap":coco_mask_ap(val_p),"test_ap":coco_mask_ap(test_p),
            "val_prepared":val_p,"test_prepared":test_p}
    return result

YOLO_EVAL=evaluate_model(YOLO_RECORDS)
MASKRCNN_EVAL=evaluate_model(MASKRCNN_RECORDS)
if not (YOLO_SCORE_FLOOR <= TRACKER_LOW_CONFIDENCE_MIN < YOLO_EVAL["threshold"]):
    raise RuntimeError(
        "Tracker low-confidence recovery threshold must be at least the YOLO inference floor "
        "and strictly below the validation-selected high-confidence threshold; got "
        f"floor={YOLO_SCORE_FLOOR}, low={TRACKER_LOW_CONFIDENCE_MIN}, "
        f"high={YOLO_EVAL['threshold']}."
    )

metric_rows=[]; threshold_rows=[]; all_frame_rows=[]; all_pair_rows=[]; geometry_rows=[]
for res in [YOLO_EVAL,MASKRCNN_EVAL]:
    for split_name in ["val","test"]:
        sm=res[f"{split_name}_summary"]; ap=res[f"{split_name}_ap"]
        metric_rows.append({"MODEL":res["model"],"SPLIT":split_name,"CONFIDENCE_THRESHOLD":res["threshold"],**sm,
                            "AP50":ap["AP50"],"AP75":ap["AP75"],"AP50_95":ap["AP50_95"],"AP_MAX_DETS":ap["AP_MAX_DETS"]})
        all_frame_rows.append(res[f"{split_name}_frames"])
        all_pair_rows.append(res[f"{split_name}_pairs"])
        g=res[f"{split_name}_geometry"].copy(); g["MODEL"]=res["model"]; g["SPLIT"]=split_name; geometry_rows.append(g)
    sweep=res["sweep"].copy(); sweep["MODEL"]=res["model"]; threshold_rows.append(sweep)

SEGMENTATION_METRICS_DF=pd.DataFrame(metric_rows)
THRESHOLD_SWEEPS_DF=pd.concat(threshold_rows,ignore_index=True)
FRAME_METRICS_DF=pd.concat(all_frame_rows,ignore_index=True)
MATCHED_DETAILS_DF=pd.concat(all_pair_rows,ignore_index=True)
GEOMETRY_METRICS_DF=pd.concat(geometry_rows,ignore_index=True)
SEGMENTATION_METRICS_DF.to_csv(OUTPUT_DIR/"fresh_segmentation_metrics.csv",index=False)
THRESHOLD_SWEEPS_DF.to_csv(OUTPUT_DIR/"fresh_validation_threshold_sweeps.csv",index=False)
FRAME_METRICS_DF.to_csv(OUTPUT_DIR/"fresh_per_frame_detection_metrics.csv",index=False)
MATCHED_DETAILS_DF.to_csv(OUTPUT_DIR/"fresh_matched_instance_details.csv",index=False)
GEOMETRY_METRICS_DF.to_csv(OUTPUT_DIR/"fresh_per_instance_geometry_metrics.csv",index=False)
print("Fresh validation-selected thresholds:", YOLO_EVAL["threshold"], MASKRCNN_EVAL["threshold"])
display(SEGMENTATION_METRICS_DF)
display(GEOMETRY_METRICS_DF)

Prepare YOLO26-L-seg:   0%|          | 0/433 [00:00<?, ?it/s]

Prepare YOLO26-L-seg:   0%|          | 0/528 [00:00<?, ?it/s]

creating index...
index created!
Loading and preparing results...
DONE (t=0.22s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *segm*
DONE (t=1.22s).
Accumulating evaluation results...
DONE (t=0.05s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.945
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.999
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.990
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = -1.000
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = -1.000
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.945
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.126
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.956
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.956
 Average Recall     (AR) @[ IoU=0

Prepare Mask R-CNN + ConvNeXt-Tiny-FPN:   0%|          | 0/433 [00:00<?, ?it/s]

Prepare Mask R-CNN + ConvNeXt-Tiny-FPN:   0%|          | 0/528 [00:00<?, ?it/s]

creating index...
index created!
Loading and preparing results...
DONE (t=0.56s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *segm*
DONE (t=2.30s).
Accumulating evaluation results...
DONE (t=0.07s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.839
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.985
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.984
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = -1.000
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = -1.000
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.839
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.111
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.879
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.890
 Average Recall     (AR) @[ IoU=0

,MODEL,SPLIT,CONFIDENCE_THRESHOLD,TP,FP,FN,precision,recall,F1,confidence_threshold,AP50,AP75,AP50_95,AP_MAX_DETS
0,YOLO26-L-seg,val,0.871056,3420,7,1,0.997957,0.999708,0.998832,0.871056,0.998843,0.989663,0.945191,100
1,YOLO26-L-seg,test,0.871056,2290,16,373,0.993062,0.859932,0.921715,0.871056,0.962095,0.947242,0.890166,100
2,Mask R-CNN + ConvNeXt-Tiny-FPN,val,0.757535,3395,152,26,0.957147,0.992400,0.974455,0.757535,0.985006,0.984418,0.838859,100
3,Mask R-CNN + ConvNeXt-Tiny-FPN,test,0.757535,2479,1780,184,0.582062,0.930905,0.716267,0.757535,0.723839,0.669741,0.570346,100


,MEASUREMENT,N,MAE,RMSE,MEAN_RELATIVE_ABS_ERROR,MEDIAN_RELATIVE_ABS_ERROR,SIGNED_BIAS,MODEL,SPLIT
0,AREA_PX,3420,13918.821053,20409.757321,0.039282,0.025144,12483.125731,YOLO26-L-seg,val
1,WIDTH_PX,3420,6.854094,10.421371,0.009052,0.005287,-3.454094,YOLO26-L-seg,val
2,HEIGHT_PX,3420,10.057018,19.540249,0.017021,0.008527,-4.186257,YOLO26-L-seg,val
3,AREA_PX,2290,10918.300000,20398.433703,0.042107,0.035012,8840.700000,YOLO26-L-seg,test
4,WIDTH_PX,2290,9.462445,20.406128,0.014888,0.009772,-4.472926,YOLO26-L-seg,test
5,HEIGHT_PX,2290,8.937118,17.781473,0.015609,0.008759,-5.757205,YOLO26-L-seg,test
6,AREA_PX,3395,13190.689249,28959.492961,0.033838,0.016854,-5329.070398,Mask R-CNN + ConvNeXt-Tiny-FPN,val
7,WIDTH_PX,3395,26.395287,37.782994,0.027711,0.019726,-15.862150,Mask R-CNN + ConvNeXt-Tiny-FPN,val
8,HEIGHT_PX,3395,30.830044,46.504717,0.047830,0.031348,-19.482180,Mask R-CNN + ConvNeXt-Tiny-FPN,val
9,AREA_PX,2479,33286.176281,88995.647693,0.069601,0.044464,-29878.176281,Mask R-CNN + ConvNeXt-Tiny-FPN,test


In [14]:
sequence_rows=[]
for res in [YOLO_EVAL,MASKRCNN_EVAL]:
    for split_name in ["val","test"]:
        prep=res[f"{split_name}_prepared"]
        for seq in sorted({int(x["frame"]["SEQUENCE_ID"]) for x in prep}):
            subset=[x for x in prep if int(x["frame"]["SEQUENCE_ID"])==seq]
            summary,frames,pairs=evaluate_at_threshold(subset,res["threshold"])
            sequence_rows.append({"MODEL":res["model"],"SPLIT":split_name,"SEQUENCE_ID":seq,**summary})
PER_SEQUENCE_DF=pd.DataFrame(sequence_rows)
PER_SEQUENCE_DF.to_csv(OUTPUT_DIR/"fresh_per_sequence_metrics.csv",index=False)

# Historical values are comparison targets, not acceptance criteria. Deviations are reported transparently.
expected = {
 "YOLO26-L-seg_test": {"AP50":0.9620952453,"AP75":0.9472423271,"AP50_95":0.8901661598,"TP":2290,"FP":16,"FN":373,
                       "AREA_PX_MEAN_REL":0.0421,"WIDTH_PX_MEAN_REL":0.0149,"HEIGHT_PX_MEAN_REL":0.0156},
 "Mask R-CNN + ConvNeXt-Tiny-FPN_test": {"AP50":0.7238387324,"AP75":0.6697410891,"AP50_95":0.5703455443,"TP":2479,"FP":1780,"FN":184,
                       "AREA_PX_MEAN_REL":0.0696,"WIDTH_PX_MEAN_REL":0.0563,"HEIGHT_PX_MEAN_REL":0.0742}
}
repro_rows=[]
for res in [YOLO_EVAL,MASKRCNN_EVAL]:
    key=res["model"]+"_test"; sm=res["test_summary"]; ap=res["test_ap"]; geo=res["test_geometry"].set_index("MEASUREMENT")
    observed={"AP50":ap["AP50"],"AP75":ap["AP75"],"AP50_95":ap["AP50_95"],"TP":sm["TP"],"FP":sm["FP"],"FN":sm["FN"],
              "AREA_PX_MEAN_REL":geo.loc["AREA_PX","MEAN_RELATIVE_ABS_ERROR"],
              "WIDTH_PX_MEAN_REL":geo.loc["WIDTH_PX","MEAN_RELATIVE_ABS_ERROR"],
              "HEIGHT_PX_MEAN_REL":geo.loc["HEIGHT_PX","MEAN_RELATIVE_ABS_ERROR"]}
    for metric,old in expected[key].items():
        val=observed[metric]
        tol=0 if metric in {"TP","FP","FN"} else 0.005
        repro_rows.append({"MODEL":res["model"],"METRIC":metric,"PRIOR_VALUE":old,"FRESH_VALUE":val,
                           "ABS_DIFFERENCE":abs(float(val)-float(old)),"TOLERANCE":tol,
                           "STATUS":"MATCH" if abs(float(val)-float(old))<=tol else "REVIEW"})
REPRODUCTION_AUDIT_DF=pd.DataFrame(repro_rows)
REPRODUCTION_AUDIT_DF.to_csv(OUTPUT_DIR/"prior_metric_reproduction_audit.csv",index=False)
display(PER_SEQUENCE_DF)
display(REPRODUCTION_AUDIT_DF)

,MODEL,SPLIT,SEQUENCE_ID,TP,FP,FN,precision,recall,F1,confidence_threshold
0,YOLO26-L-seg,val,2,1638,7,0,0.995745,1.000000,0.997868,0.871056
1,YOLO26-L-seg,val,4,1782,0,1,1.000000,0.999439,0.999719,0.871056
2,YOLO26-L-seg,test,1,933,10,81,0.989396,0.920118,0.953500,0.871056
3,YOLO26-L-seg,test,3,1357,6,292,0.995598,0.822923,0.901062,0.871056
4,Mask R-CNN + ConvNeXt-Tiny-FPN,val,2,1634,11,4,0.993313,0.997558,0.995431,0.757535
5,Mask R-CNN + ConvNeXt-Tiny-FPN,val,4,1761,141,22,0.925868,0.987661,0.955767,0.757535
6,Mask R-CNN + ConvNeXt-Tiny-FPN,test,1,971,623,43,0.609159,0.957594,0.744632,0.757535
7,Mask R-CNN + ConvNeXt-Tiny-FPN,test,3,1508,1157,141,0.565854,0.914494,0.699119,0.757535


,MODEL,METRIC,PRIOR_VALUE,FRESH_VALUE,ABS_DIFFERENCE,TOLERANCE,STATUS
0,YOLO26-L-seg,AP50,0.962095,0.962095,3.719725e-11,0.005,MATCH
1,YOLO26-L-seg,AP75,0.947242,0.947242,1.446510e-12,0.005,MATCH
2,YOLO26-L-seg,AP50_95,0.890166,0.890166,2.807909e-11,0.005,MATCH
3,YOLO26-L-seg,TP,2290.000000,2290.000000,0.000000e+00,0.000,MATCH
4,YOLO26-L-seg,FP,16.000000,16.000000,0.000000e+00,0.000,MATCH
5,YOLO26-L-seg,FN,373.000000,373.000000,0.000000e+00,0.000,MATCH
6,YOLO26-L-seg,AREA_PX_MEAN_REL,0.042100,0.042107,6.882427e-06,0.005,MATCH
7,YOLO26-L-seg,WIDTH_PX_MEAN_REL,0.014900,0.014888,1.228396e-05,0.005,MATCH
8,YOLO26-L-seg,HEIGHT_PX_MEAN_REL,0.015600,0.015609,9.043978e-06,0.005,MATCH
9,Mask R-CNN + ConvNeXt-Tiny-FPN,AP50,0.723839,0.723839,7.684742e-12,0.005,MATCH


## Historical tracker-comparison diagnostics

This inference notebook retains the earlier four-configuration tracker comparisons as part of the development evidence. They are useful for tracing the implementation history, but the test sequences were inspected during development and these comparisons are retrospective. They are not the final tracking result reported in the paper. For the final frozen tracker, whole-track purity analysis, strict growth histories, and temporal growth metrics, run `tracking-and-growth-history.ipynb` using the cached YOLO predictions exported here. Its custom frame-matching MOTA is not standard CLEAR-MOTA/TrackEval.

In [15]:
def rle_to_coco(instance): return coco_rle(instance)


def baseline_iou_matches(iou,min_iou):
    values=np.asarray(iou,dtype=np.float64); n,m=values.shape
    if n==0 or m==0: return []
    valid=values>=float(min_iou); bonus=float(min(n,m)+1)
    weights=np.zeros((n,m+n),dtype=np.float64); weights[:,:m]=np.where(valid,bonus+values,0.0)
    rows,cols=hungarian_min_cost(-weights)
    return [(int(r),int(c),float(values[r,c])) for r,c in zip(rows,cols) if c<m and valid[r,c]]


def association_matches(track_states,track_ids,detections,use_spatial,max_distance_px):
    if not track_ids or not detections: return []
    old_masks=[track_states[t]["last_mask"] for t in track_ids]
    new_masks=[rle_to_coco(d) for d in detections]
    iou=mask_utils.iou(old_masks,new_masks,[0]*len(new_masks)).astype(np.float64)
    if not use_spatial:
        return [(ti,di,overlap,np.nan,np.nan) for ti,di,overlap in baseline_iou_matches(iou,TRACKER_ASSOCIATION_IOU)]
    old_xy=np.asarray([[track_states[t]["centroid_x"],track_states[t]["centroid_y"]] for t in track_ids],dtype=np.float64)
    new_xy=np.asarray([[float(d["CENTROID_X_PX"]),float(d["CENTROID_Y_PX"])] for d in detections],dtype=np.float64)
    dist=np.linalg.norm(old_xy[:,None,:]-new_xy[None,:,:],axis=2)
    valid=(dist<=float(max_distance_px))
    norm_dist=np.clip(dist/max(float(max_distance_px),1.0),0.0,1.0)
    cost=0.5*(1.0-iou)+0.5*norm_dist
    valid &= cost < TRACKER_SPATIAL_MAX_COST
    augmented=np.full((len(track_ids),len(detections)+len(track_ids)),1e6,dtype=np.float64)
    augmented[:,:len(detections)]=np.where(valid,cost,1e6)
    augmented[:,len(detections):]=TRACKER_SPATIAL_MAX_COST
    rows,cols=linear_sum_assignment(augmented)
    return [(int(r),int(c),float(iou[r,c]),float(dist[r,c]),float(cost[r,c])) for r,c in zip(rows,cols) if c<len(detections) and valid[r,c]]


def make_track_observation(frame,det,tid,stage,overlap=np.nan,distance=np.nan,cost=np.nan,offset=0):
    return {"SEQUENCE_ID":int(frame["SEQUENCE_ID"]),"FRAME_ID":int(frame["FRAME_ID"]),"FRAME_KEY":frame["FRAME_KEY"],
            "SPLIT":frame["SPLIT"],"VALIDATION_OFFSET":int(offset),"ELAPSED_MINUTES_NOMINAL":float(frame["ELAPSED_MINUTES_NOMINAL"]),
            "FRAME_INSTANCE_ID":int(det["FRAME_INSTANCE_ID"]),"CLUSTER_TRACK_ID":int(tid),
            "CONFIDENCE":float(det["CONFIDENCE"]),"AREA_PX":int(det["AREA_PX"]),"WIDTH_PX":int(det["WIDTH_PX"]),
            "HEIGHT_PX":int(det["HEIGHT_PX"]),"SIZE_INDEX_PCT":float(det["SIZE_INDEX_PCT"]),
            "CENTROID_X_PX":float(det["CENTROID_X_PX"]),"CENTROID_Y_PX":float(det["CENTROID_Y_PX"]),
            "ASSOCIATION_STAGE":stage,"ASSOCIATION_MASK_IOU":float(overlap) if np.isfinite(overlap) else np.nan,
            "ASSOCIATION_CENTROID_DISTANCE_PX":float(distance) if np.isfinite(distance) else np.nan,
            "ASSOCIATION_COST":float(cost) if np.isfinite(cost) else np.nan}


def run_tracker(frame_records,use_spatial=False,use_low_confidence=False,high_threshold=None):
    high_threshold=float(YOLO_EVAL["threshold"] if high_threshold is None else high_threshold)
    groups=defaultdict(list)
    for fr in frame_records: groups[(int(fr["SEQUENCE_ID"]),int(fr.get("_EVAL_OFFSET",0)))].append(fr)
    observations=[]; summaries=[]; events=[]
    for (seq,offset),frames in sorted(groups.items()):
        frames.sort(key=lambda x:int(x["FRAME_ID"]))
        states={}; active=set(); next_id=1; seq_obs=[]
        for ordinal,fr in enumerate(frames):
            preds=fr["PREDICTIONS"]
            high=[p for p in preds if float(p["CONFIDENCE"])>=high_threshold]
            low=[p for p in preds if TRACKER_LOW_CONFIDENCE_MIN<=float(p["CONFIDENCE"])<high_threshold] if use_low_confidence else []
            eligible=sorted(t for t in active if ordinal-states[t]["last_ordinal"]<=TRACKER_MAX_GAP+1)
            matched_tracks=set(); used_high=set()
            if eligible and high:
                for ti,di,overlap,dist,cost in association_matches(states,eligible,high,use_spatial,TRACKER_SPATIAL_DISTANCE_PX):
                    tid=eligible[ti]; det=high[di]; gap=ordinal-states[tid]["last_ordinal"]-1
                    state=states[tid]
                    state.update({"last_mask":rle_to_coco(det),"last_ordinal":ordinal,
                                  "centroid_x":float(det["CENTROID_X_PX"]),"centroid_y":float(det["CENTROID_Y_PX"]),
                                  "observation_count":state["observation_count"]+1})
                    ob=make_track_observation(fr,det,tid,"high_confidence",overlap,dist,cost,offset)
                    observations.append(ob); seq_obs.append(ob); matched_tracks.add(tid); used_high.add(di)
                    if gap>0: events.append({"SEQUENCE_ID":seq,"VALIDATION_OFFSET":offset,"FRAME_ID":fr["FRAME_ID"],"EVENT":"reassociated_after_gap","CLUSTER_TRACK_ID":tid,"GAP_OBSERVATIONS":gap})
            if use_low_confidence and low:
                remaining=[t for t in eligible if t not in matched_tracks]
                if remaining:
                    for ti,di,overlap,dist,cost in association_matches(states,remaining,low,use_spatial,TRACKER_SPATIAL_DISTANCE_PX):
                        tid=remaining[ti]; det=low[di]; gap=ordinal-states[tid]["last_ordinal"]-1
                        state=states[tid]
                        state.update({"last_mask":rle_to_coco(det),"last_ordinal":ordinal,
                                      "centroid_x":float(det["CENTROID_X_PX"]),"centroid_y":float(det["CENTROID_Y_PX"]),
                                      "observation_count":state["observation_count"]+1})
                        ob=make_track_observation(fr,det,tid,"low_confidence_recovery",overlap,dist,cost,offset)
                        observations.append(ob); seq_obs.append(ob); matched_tracks.add(tid)
                        if gap>0: events.append({"SEQUENCE_ID":seq,"VALIDATION_OFFSET":offset,"FRAME_ID":fr["FRAME_ID"],"EVENT":"reassociated_after_gap","CLUSTER_TRACK_ID":tid,"GAP_OBSERVATIONS":gap})
            for tid in list(active):
                if tid not in matched_tracks and ordinal-states[tid]["last_ordinal"]>TRACKER_MAX_GAP:
                    states[tid]["closed"]=True; active.remove(tid)
            for di,det in enumerate(high):
                if di in used_high: continue
                tid=next_id; next_id+=1
                states[tid]={"last_mask":rle_to_coco(det),"last_ordinal":ordinal,
                             "centroid_x":float(det["CENTROID_X_PX"]),"centroid_y":float(det["CENTROID_Y_PX"]),
                             "observation_count":1,"closed":False}
                active.add(tid)
                ob=make_track_observation(fr,det,tid,"new_high_confidence",offset=offset)
                observations.append(ob); seq_obs.append(ob)
        grouped=defaultdict(list)
        for row in seq_obs: grouped[int(row["CLUSTER_TRACK_ID"])].append(row)
        for tid,rows in grouped.items():
            rows.sort(key=lambda x:int(x["FRAME_ID"]))
            first,last=rows[0],rows[-1]
            summaries.append({"SEQUENCE_ID":seq,"VALIDATION_OFFSET":offset,"CLUSTER_TRACK_ID":tid,
                              "FIRST_FRAME_ID":first["FRAME_ID"],"LAST_OBSERVED_FRAME_ID":last["FRAME_ID"],
                              "OBSERVATION_COUNT":len(rows),"FIRST_AREA_PX":first["AREA_PX"],"LAST_AREA_PX":last["AREA_PX"],
                              "AREA_CHANGE_PX":last["AREA_PX"]-first["AREA_PX"]})
    obs=pd.DataFrame(observations)
    if not obs.empty:
        obs=obs.sort_values(["SEQUENCE_ID","VALIDATION_OFFSET","CLUSTER_TRACK_ID","FRAME_ID"]).reset_index(drop=True)
        g=obs.groupby(["SEQUENCE_ID","VALIDATION_OFFSET","CLUSTER_TRACK_ID"])
        prev=g["AREA_PX"].shift(1); prev_time=g["ELAPSED_MINUTES_NOMINAL"].shift(1)
        dt=((obs["ELAPSED_MINUTES_NOMINAL"]-prev_time)/60.0).where(lambda x:x>0)
        delta=obs["AREA_PX"]-prev
        obs["AREA_CHANGE_PX"]=delta
        obs["AREA_CHANGE_PX_PER_HOUR"]=delta/dt
        obs["AREA_RELATIVE_CHANGE"]=delta/prev.where(prev>0)
        obs["AREA_RELATIVE_CHANGE_PER_HOUR"]=(delta/prev.where(prev>0))/dt
    return {"observations":obs,"summary":pd.DataFrame(summaries),"events":pd.DataFrame(events)}


def evaluate_tracking_identity(track_result,eval_records,split_label):
    obs=track_result["observations"].copy()
    obs_lookup=defaultdict(list)
    if not obs.empty:
        for row in obs.to_dict("records"):
            obs_lookup[(int(row["SEQUENCE_ID"]),int(row["FRAME_ID"]))].append(row)
    total_gt=sum(len(fr["GROUND_TRUTH_INSTANCES"]) for fr in eval_records)
    total_pred=len(obs); match_rows=[]; gt_state_rows=[]
    for fr in eval_records:
        seq,frame=int(fr["SEQUENCE_ID"]),int(fr["FRAME_ID"]); offset=int(fr.get("_EVAL_OFFSET",0))
        frame_obs=obs_lookup.get((seq,frame),[])
        pred_by_instance={int(p["FRAME_INSTANCE_ID"]):p for p in fr["PREDICTIONS"]}
        track_rows=[]
        for o in frame_obs:
            det=pred_by_instance.get(int(o["FRAME_INSTANCE_ID"]))
            if det is None: raise KeyError(f"Tracked instance not in frame predictions: {(seq,frame,o['FRAME_INSTANCE_ID'])}")
            track_rows.append((o,det))
        gts=fr["GROUND_TRUTH_INSTANCES"]
        iou=frame_iou_matrix([d for _,d in track_rows],gts) if track_rows and gts else np.zeros((len(track_rows),len(gts)))
        pairs=max_cardinality_iou_matches(iou,0.50)
        frame_pred_to_gt={}
        for pi,gi,overlap in pairs:
            o,det=track_rows[pi]; gt=gts[gi]
            pred_key=f"{seq}:{offset}:{int(o['CLUSTER_TRACK_ID'])}"; gt_key=f"{seq}:{offset}:{gt['GT_TRACK_ID']}"
            frame_pred_to_gt[int(gt["GT_FRAME_INSTANCE_ID"])]=pred_key
            match_rows.append({"SPLIT":split_label,"SEQUENCE_ID":seq,"FRAME_ID":frame,"VALIDATION_OFFSET":offset,
                               "FRAME_INSTANCE_ID":int(o["FRAME_INSTANCE_ID"]),"CLUSTER_TRACK_ID":int(o["CLUSTER_TRACK_ID"]),
                               "GT_FRAME_INSTANCE_ID":int(gt["GT_FRAME_INSTANCE_ID"]),"GT_TRACK_ID":str(gt["GT_TRACK_ID"]),
                               "PRED_TRACK_KEY":pred_key,"GT_TRACK_KEY":gt_key,"MASK_IOU":float(overlap),
                               "CONFIDENCE":float(det["CONFIDENCE"]),"ASSOCIATION_STAGE":o["ASSOCIATION_STAGE"]})
        for gt in gts:
            gt_key=f"{seq}:{offset}:{gt['GT_TRACK_ID']}"
            gt_state_rows.append({"SEQUENCE_ID":seq,"FRAME_ID":frame,"VALIDATION_OFFSET":offset,
                                  "GT_FRAME_INSTANCE_ID":int(gt["GT_FRAME_INSTANCE_ID"]),"GT_TRACK_ID":str(gt["GT_TRACK_ID"]),
                                  "GT_TRACK_KEY":gt_key,"MATCHED_PRED_TRACK_KEY":frame_pred_to_gt.get(int(gt["GT_FRAME_INSTANCE_ID"]))})
    matches=pd.DataFrame(match_rows); states=pd.DataFrame(gt_state_rows)
    det_tp=len(matches); det_fp=total_pred-det_tp; det_fn=total_gt-det_tp
    if matches.empty: idtp=idsw=0
    else:
        cont=matches.groupby(["PRED_TRACK_KEY","GT_TRACK_KEY"]).size().unstack(fill_value=0)
        matrix=cont.to_numpy(dtype=np.int64); rr,cc=linear_sum_assignment(-matrix); idtp=int(matrix[rr,cc].sum())
        idsw=0
        for _,g in matches.sort_values(["SEQUENCE_ID","VALIDATION_OFFSET","GT_TRACK_KEY","FRAME_ID"]).groupby("GT_TRACK_KEY"):
            p=g.PRED_TRACK_KEY.tolist(); idsw+=sum(a!=b for a,b in zip(p,p[1:]))
    idfp=total_pred-idtp; idfn=total_gt-idtp
    den=2*idtp+idfp+idfn; idf1=2*idtp/den if den else 0.0
    idp=idtp/(idtp+idfp) if idtp+idfp else 0.0; idr=idtp/(idtp+idfn) if idtp+idfn else 0.0
    mota_custom=1.0-(det_fp+det_fn+idsw)/total_gt if total_gt else 0.0
    per_gt=[]
    if not states.empty:
        for key,g in states.sort_values("FRAME_ID").groupby("GT_TRACK_KEY"):
            pred=g.MATCHED_PRED_TRACK_KEY.tolist(); distinct=sorted({x for x in pred if pd.notna(x)})
            fragments=switches=0; seen=False; missing_after=False; last=None
            for p in pred:
                matched=pd.notna(p)
                if matched:
                    if seen and missing_after: fragments+=1
                    if last is not None and p!=last: switches+=1
                    last=p; seen=True; missing_after=False
                elif seen: missing_after=True
            per_gt.append({"GT_TRACK_KEY":key,"SEQUENCE_ID":int(g.SEQUENCE_ID.iloc[0]),"GT_TRACK_ID":str(g.GT_TRACK_ID.iloc[0]),
                           "GT_OBSERVATIONS":len(g),"MATCHED_OBSERVATIONS":int(g.MATCHED_PRED_TRACK_KEY.notna().sum()),
                           "PRED_TRACK_IDS_ASSOCIATED":len(distinct),"PRED_TRACK_KEYS":";".join(distinct),
                           "ID_SWITCHES":switches,"FRAGMENTATIONS":fragments})
    per_gt=pd.DataFrame(per_gt)
    low_total=int((obs.ASSOCIATION_STAGE=="low_confidence_recovery").sum()) if not obs.empty else 0
    low_matched=int((matches.ASSOCIATION_STAGE=="low_confidence_recovery").sum()) if not matches.empty else 0
    summary={"SPLIT":split_label,"IDTP":idtp,"IDFP":idfp,"IDFN":idfn,"IDF1":idf1,"IDP":idp,"IDR":idr,
             "MOTA_CUSTOM_FRAME_MATCH":mota_custom,"IDSW":idsw,"DETECTION_TP":det_tp,"DETECTION_FP":det_fp,"DETECTION_FN":det_fn,
             "GT_DETECTIONS":total_gt,"TRACKED_PREDICTIONS":total_pred,"GT_PHYSICAL_IDENTITIES":int(per_gt[["SEQUENCE_ID","GT_TRACK_ID"]].drop_duplicates().shape[0]) if not per_gt.empty else 0,
             "GT_IDENTITY_STREAMS_FRAGMENTED":int((per_gt.PRED_TRACK_IDS_ASSOCIATED>1).sum()) if not per_gt.empty else 0,
             "TOTAL_IDENTITY_STREAM_FRAGMENTATIONS":int(per_gt.FRAGMENTATIONS.sum()) if not per_gt.empty else 0,
             "PREDICTED_TRACKS":int(len(track_result["summary"])),
             "SINGLETON_PREDICTED_TRACKS":int((track_result["summary"].OBSERVATION_COUNT==1).sum()) if not track_result["summary"].empty else 0,
             "LOW_CONFIDENCE_RECOVERY_OBSERVATIONS":low_total,"LOW_CONFIDENCE_RECOVERY_MATCHED":low_matched,
             "LOW_CONFIDENCE_RECOVERY_UNMATCHED":low_total-low_matched,
             "LOW_CONFIDENCE_RECOVERY_PRECISION":low_matched/low_total if low_total else np.nan}
    return summary,matches,per_gt,states

# Prepare eight disjoint validation streams: all 433 frames are used once, but tracks do not cross phase offsets.
val_full=[dict(r) for r in YOLO_RECORDS if r["SPLIT"]=="val"]
val_full.sort(key=lambda r:(int(r["SEQUENCE_ID"]),int(r["FRAME_ID"])))
val_records=[]
for seq in sorted({int(r["SEQUENCE_ID"]) for r in val_full}):
    seq_rows=[r for r in val_full if int(r["SEQUENCE_ID"])==seq]
    for position,fr in enumerate(seq_rows):
        tagged=dict(fr); tagged["_EVAL_OFFSET"]=int(position%VALIDATION_CADENCE_STRIDE); val_records.append(tagged)
assert len(val_records)==433 and len({(r["SEQUENCE_ID"],r["FRAME_ID"]) for r in val_records})==433

test_records=[dict(r,_EVAL_OFFSET=0) for r in YOLO_RECORDS if r["SPLIT"]=="test"]
test_records.sort(key=lambda r:(int(r["SEQUENCE_ID"]),int(r["FRAME_ID"])))

tracker_variants={
 "A_iou_only":{"use_spatial":False,"use_low_confidence":False},
 "B_spatial_only":{"use_spatial":True,"use_low_confidence":False},
 "C_iou_plus_lowconf":{"use_spatial":False,"use_low_confidence":True},
 "D_spatial_plus_lowconf":{"use_spatial":True,"use_low_confidence":True},
}
tracking_rows=[]; tracking_details={}
for variant,cfg in tracker_variants.items():
    for split_name,frames in [("validation_8_offsets",val_records),("test_retrospective",test_records)]:
        tr=run_tracker(frames,**cfg)
        met,matches,per_gt,states=evaluate_tracking_identity(tr,frames,split_name)
        met.update({"VARIANT":variant,"USE_SPATIAL":cfg["use_spatial"],"USE_LOW_CONFIDENCE":cfg["use_low_confidence"],
                    "HIGH_CONFIDENCE_THRESHOLD":YOLO_EVAL["threshold"],"LOW_CONFIDENCE_MIN":TRACKER_LOW_CONFIDENCE_MIN if cfg["use_low_confidence"] else None,
                    "MAX_GAP":TRACKER_MAX_GAP,"SPATIAL_DISTANCE_PX":TRACKER_SPATIAL_DISTANCE_PX if cfg["use_spatial"] else None})
        tracking_rows.append(met)
        safe=f"{variant}_{'validation' if split_name.startswith('validation') else 'test'}"
        tr["observations"].to_csv(OUTPUT_DIR/f"{safe}_tracked_observations.csv",index=False)
        tr["summary"].to_csv(OUTPUT_DIR/f"{safe}_track_summary.csv",index=False)
        if not matches.empty: matches.to_csv(OUTPUT_DIR/f"{safe}_frame_matches.csv",index=False)
        if not per_gt.empty: per_gt.to_csv(OUTPUT_DIR/f"{safe}_per_gt_identity.csv",index=False)
        if variant == SELECTED_TRACKING_VARIANT and split_name == "test_retrospective":
            if "selected" in tracking_details:
                raise RuntimeError(
                    f"Selected tracker {SELECTED_TRACKING_VARIANT!r} was captured more than once; "
                    "check the split/variant loop."
                )
            tracking_details["selected"] = {
                "variant": variant,
                "split": split_name,
                "tracker": tr,
                "metrics": met,
                "matches": matches,
                "per_gt": per_gt,
                "gt_states": states,
            }

if "selected" not in tracking_details:
    raise RuntimeError(
        "Selected tracker result was not captured. Expected exactly one run with "
        f"VARIANT={SELECTED_TRACKING_VARIANT!r} and SPLIT='test_retrospective'. "
        f"Captured keys: {list(tracking_details)}"
    )

TRACKING_METRICS_DF=pd.DataFrame(tracking_rows)
expected_tracking_rows = len(tracker_variants) * 2
if len(TRACKING_METRICS_DF) != expected_tracking_rows:
    raise RuntimeError(
        f"Expected {expected_tracking_rows} tracking result rows "
        f"({len(tracker_variants)} variants × validation/test); got {len(TRACKING_METRICS_DF)}."
    )
if TRACKING_METRICS_DF.duplicated(["VARIANT", "SPLIT"]).any():
    raise RuntimeError("Duplicate tracker variant/split result rows detected.")
TRACKING_METRICS_DF.to_csv(OUTPUT_DIR/"fresh_tracking_2x2_metrics.csv",index=False)
display(TRACKING_METRICS_DF)
print("Interpretation: validation confirms the frozen variants on eight disjoint stride-8 streams; test results remain retrospective because the test set was previously inspected.")

,SPLIT,IDTP,IDFP,IDFN,IDF1,IDP,IDR,MOTA_CUSTOM_FRAME_MATCH,IDSW,DETECTION_TP,...,LOW_CONFIDENCE_RECOVERY_MATCHED,LOW_CONFIDENCE_RECOVERY_UNMATCHED,LOW_CONFIDENCE_RECOVERY_PRECISION,VARIANT,USE_SPATIAL,USE_LOW_CONFIDENCE,HIGH_CONFIDENCE_THRESHOLD,LOW_CONFIDENCE_MIN,MAX_GAP,SPATIAL_DISTANCE_PX
0,validation_8_offsets,3420,7,1,0.998832,0.997957,0.999708,0.997662,0,3420,...,0,0,NaN,A_iou_only,False,False,0.871056,NaN,0,NaN
1,test_retrospective,1846,460,817,0.743007,0.800520,0.693203,0.832895,56,2290,...,0,0,NaN,A_iou_only,False,False,0.871056,NaN,0,NaN
2,validation_8_offsets,3391,36,30,0.990362,0.989495,0.991231,0.995323,8,3420,...,0,0,NaN,B_spatial_only,True,False,0.871056,NaN,0,47.0
3,test_retrospective,1900,406,763,0.764741,0.823938,0.713481,0.830267,63,2290,...,0,0,NaN,B_spatial_only,True,False,0.871056,NaN,0,47.0
4,validation_8_offsets,3420,7,1,0.998832,0.997957,0.999708,0.997662,0,3420,...,0,0,NaN,C_iou_plus_lowconf,False,True,0.871056,0.6,0,NaN
5,test_retrospective,2187,262,476,0.855634,0.893018,0.821254,0.891851,6,2415,...,125,18,0.874126,C_iou_plus_lowconf,False,True,0.871056,0.6,0,NaN
6,validation_8_offsets,3391,36,30,0.990362,0.989495,0.991231,0.995323,8,3420,...,0,0,NaN,D_spatial_plus_lowconf,True,True,0.871056,0.6,0,47.0
7,test_retrospective,2194,253,469,0.858708,0.896608,0.823883,0.887721,15,2413,...,123,18,0.872340,D_spatial_plus_lowconf,True,True,0.871056,0.6,0,47.0


Interpretation: validation confirms the frozen variants on eight disjoint stride-8 streams; test results remain retrospective because the test set was previously inspected.


In [16]:
selected = tracking_details.get("selected")
if selected is None:
    raise RuntimeError(
        "No selected tracker result is available. Re-run the tracker cell and verify "
        f"the selected variant/split ({SELECTED_TRACKING_VARIANT!r}, 'test_retrospective')."
    )
if selected.get("variant") != SELECTED_TRACKING_VARIANT or selected.get("split") != "test_retrospective":
    raise RuntimeError(
        "Selected tracker metadata does not match the requested variant/split: "
        f"{selected.get('variant')!r}/{selected.get('split')!r}."
    )

selected_obs = selected["tracker"]["observations"].copy()
selected_matches = selected["matches"].copy()
required_match_cols = {
    "SEQUENCE_ID", "FRAME_ID", "FRAME_INSTANCE_ID", "GT_TRACK_ID", "GT_TRACK_KEY", "PRED_TRACK_KEY"
}
if not selected_matches.empty:
    missing = required_match_cols - set(selected_matches.columns)
    if missing:
        raise RuntimeError(f"Selected tracker match table is missing columns: {sorted(missing)}")
    id_link = selected_matches[list(sorted(required_match_cols))].drop_duplicates()
    if selected_obs.empty:
        raise RuntimeError("Selected tracker has matched detections but an empty observation table.")
    selected_obs = selected_obs.merge(
        id_link,
        on=["SEQUENCE_ID", "FRAME_ID", "FRAME_INSTANCE_ID"],
        how="left",
        validate="one_to_one",
    )
else:
    # Preserve an explicit output schema even if a future run has zero identity matches.
    for column in ["GT_TRACK_ID", "GT_TRACK_KEY", "PRED_TRACK_KEY"]:
        if column not in selected_obs.columns:
            selected_obs[column] = pd.Series(dtype="object")

selected_obs.to_csv(OUTPUT_DIR / "selected_yolo_growth_histories_with_source_ids.csv", index=False)
print(
    "Selected tracker:", selected["variant"], selected["split"],
    "| observations:", len(selected_obs), "| identity-matched:",
    int(selected_obs["GT_TRACK_ID"].notna().sum()) if "GT_TRACK_ID" in selected_obs else 0,
)

# Export every GT cluster's observation-level size-reference series separately from predicted track histories.
gt_history_rows = []
for fr in YOLO_RECORDS:
    for gt in fr["GROUND_TRUTH_INSTANCES"]:
        gt_history_rows.append({
            "SEQUENCE_ID": fr["SEQUENCE_ID"], "FRAME_ID": fr["FRAME_ID"], "SPLIT": fr["SPLIT"],
            "GT_TRACK_ID": gt["GT_TRACK_ID"], "COCO_ANNOTATION_ID": gt["COCO_ANNOTATION_ID"],
            "AREA_PX": gt["AREA_PX"], "WIDTH_PX": gt["WIDTH_PX"], "HEIGHT_PX": gt["HEIGHT_PX"],
            "SIZE_INDEX_PCT": gt["SIZE_INDEX_PCT"],
            "ELAPSED_MINUTES_NOMINAL": fr["ELAPSED_MINUTES_NOMINAL"],
        })
GT_GROWTH_HISTORY_DF = pd.DataFrame(gt_history_rows)
GT_GROWTH_HISTORY_DF.to_csv(OUTPUT_DIR / "source_gt_cluster_growth_histories.csv", index=False)

# Descriptive maturity-label size distribution. The label is attached to frame observations only.
maturity_rows = []
for fr in YOLO_RECORDS:
    for gt in fr["GROUND_TRUTH_INSTANCES"]:
        maturity_rows.append({
            "SPLIT": fr["SPLIT"], "ORIGINAL_CATEGORY_NAME": gt["ORIGINAL_CATEGORY_NAME"],
            "AREA_PX": gt["AREA_PX"], "SIZE_INDEX_PCT": gt["SIZE_INDEX_PCT"],
        })
MATURITY_DF = pd.DataFrame(maturity_rows)
MATURITY_SUMMARY = MATURITY_DF.groupby(["SPLIT", "ORIGINAL_CATEGORY_NAME"]).agg(
    ANNOTATION_OBSERVATIONS=("AREA_PX", "size"), MEAN_AREA_PX=("AREA_PX", "mean"),
    MEDIAN_AREA_PX=("AREA_PX", "median"), MEAN_SIZE_INDEX_PCT=("SIZE_INDEX_PCT", "mean"),
    MEDIAN_SIZE_INDEX_PCT=("SIZE_INDEX_PCT", "median"),
).reset_index()
MATURITY_SUMMARY.to_csv(OUTPUT_DIR / "fresh_size_index_by_maturity_label.csv", index=False)
display(MATURITY_SUMMARY)


Selected tracker: C_iou_plus_lowconf test_retrospective | observations: 2449 | identity-matched: 2415


,SPLIT,ORIGINAL_CATEGORY_NAME,ANNOTATION_OBSERVATIONS,MEAN_AREA_PX,MEDIAN_AREA_PX,MEAN_SIZE_INDEX_PCT,MEDIAN_SIZE_INDEX_PCT
0,test,immature_cluster,812,113243.504926,72119.5,16.916049,10.773042
1,test,mature_cluster,948,529641.165612,366372.5,79.116556,54.727866
2,test,overstayed_cluster,903,282378.099668,124417.0,42.180979,18.585120
3,val,immature_cluster,1206,223443.732172,204699.0,33.377501,30.577457
4,val,mature_cluster,577,601224.672444,573944.0,89.809533,85.734410
5,val,overstayed_cluster,1638,967024.238095,812068.0,144.451815,121.304815


## Figures and assignment verification

The figures are generated from the fresh predictions and original source frames. Source images are resolved on demand from the same sequence loader; a separate thumbnail archive is not required. The assignment implementation is also compared with SciPy on random rectangular, tied-cost, and empty matrices.

In [17]:
# Map frame-level metrics and fresh records for source-traceable visual selection.
YOLO_BY_KEY={r["FRAME_KEY"]:r for r in YOLO_RECORDS}
MASK_BY_KEY={r["FRAME_KEY"]:r for r in MASKRCNN_RECORDS}
FRAME_ERRORS=FRAME_METRICS_DF.copy(); FRAME_ERRORS["TOTAL_ERRORS"]=FRAME_ERRORS.FP+FRAME_ERRORS.FN


def decode_rle(mask_json):
    return mask_utils.decode({"size":[int(x) for x in mask_json["size"]],"counts":mask_json["counts"].encode("ascii")}).astype(np.uint8)


# Source frames are deliberately not archived as thumbnails. Resolve them from the original
# sequence data only when a publication figure needs them.
_VISUAL_SOURCE_STATE = {}
_VISUAL_FRAME_LOOKUP = {
    frame["FRAME_KEY"]: frame
    for seq in [1, 2, 3, 4]
    for frame in SEQUENCE_DATA[seq]["eval_frames"]
}


def load_source_image_rgb(frame_key, max_width=1600):
    frame = _VISUAL_FRAME_LOOKUP[frame_key]
    seq = int(frame["SEQUENCE_ID"])
    image_path = Path(frame["IMAGE_PATH"])
    if not image_path.is_file():
        if seq not in _VISUAL_SOURCE_STATE:
            root, source_mode, archive_sha = get_sequence_root(seq)
            tracked_candidates = [
                candidate for candidate in Path(root).rglob("*_annotations_tracked.json")
                if infer_sequence(candidate) == seq or SEQUENCE_ARCHIVES[seq].replace(".zip", "") in candidate.name.lower()
            ]
            if len(tracked_candidates) != 1:
                raise RuntimeError(f"Sequence {seq}: expected one tracked annotation JSON when resolving figures; found {tracked_candidates[:5]}")
            tracked_path = tracked_candidates[0]
            image_index = build_image_index(root)
            _VISUAL_SOURCE_STATE[seq] = {
                "root": Path(root), "source_mode": source_mode,
                "tracked_path": tracked_path, "image_index": image_index
            }
        state = _VISUAL_SOURCE_STATE[seq]
        image_path = resolve_image_path(
            frame["FILE_NAME"], state["tracked_path"], state["root"], state["image_index"], seq
        )
        frame["IMAGE_PATH"] = str(image_path)
    image = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
    if image is None:
        raise FileNotFoundError(f"Could not read source image for figure {frame_key}: {image_path}")
    if max_width and image.shape[1] > max_width:
        scale = max_width / image.shape[1]
        image = cv2.resize(image, (int(round(image.shape[1] * scale)), int(round(image.shape[0] * scale))), interpolation=cv2.INTER_AREA)
    return cv2.cvtColor(image, cv2.COLOR_BGR2RGB)


def cleanup_visual_source_extractions():
    for seq, state in list(_VISUAL_SOURCE_STATE.items()):
        if state["source_mode"] in {"attached_archive", "google_drive"}:
            shutil.rmtree(state["root"], ignore_errors=True)
    _VISUAL_SOURCE_STATE.clear()


def resize_mask(mask, out_w, out_h):
    if mask.shape==(out_h,out_w): return mask
    return cv2.resize(mask,(out_w,out_h),interpolation=cv2.INTER_NEAREST)


def mask_overlay_panel(image_rgb, instances, color=(0,220,0), alpha=0.32, contour=True):
    canvas=image_rgb.copy(); h,w=canvas.shape[:2]
    for inst in instances:
        mask=decode_rle(inst["MASK_RLE"])
        small=resize_mask(mask,w,h).astype(bool)
        if small.any():
            color_arr=np.asarray(color,dtype=np.uint8)
            canvas[small]=((1-alpha)*canvas[small]+alpha*color_arr).astype(np.uint8)
            if contour:
                cnts,_=cv2.findContours(small.astype(np.uint8),cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)
                cv2.drawContours(canvas,cnts,-1,color,1)
    return canvas


def error_code_overlay(image_rgb, predictions, ground_truth):
    """TP predictions=cyan, unmatched predictions=red, missed GT instances=yellow."""
    iou = frame_iou_matrix(predictions, ground_truth) if predictions and ground_truth else np.zeros((len(predictions), len(ground_truth)))
    pairs = max_cardinality_iou_matches(iou, MASK_MATCH_IOU)
    matched_pred = {p for p, _, _ in pairs}
    matched_gt = {g for _, g, _ in pairs}
    canvas = image_rgb.copy()
    tp = [predictions[i] for i in sorted(matched_pred)]
    fp = [predictions[i] for i in range(len(predictions)) if i not in matched_pred]
    fn = [ground_truth[i] for i in range(len(ground_truth)) if i not in matched_gt]
    canvas = mask_overlay_panel(canvas, tp, color=(0, 205, 220), alpha=0.38)
    canvas = mask_overlay_panel(canvas, fp, color=(230, 45, 45), alpha=0.42)
    canvas = mask_overlay_panel(canvas, fn, color=(255, 210, 0), alpha=0.42)
    return canvas, len(tp), len(fp), len(fn)


def save_comparison_figure(frame_key, suffix):
    y=YOLO_BY_KEY[frame_key]; m=MASK_BY_KEY[frame_key]
    original=load_source_image_rgb(frame_key)
    gt=mask_overlay_panel(original,y["GROUND_TRUTH_INSTANCES"],color=(20,190,30),alpha=0.32)
    yp=[p for p in y["PREDICTIONS"] if p["CONFIDENCE"]>=YOLO_EVAL["threshold"]]
    mp=[p for p in m["PREDICTIONS"] if p["CONFIDENCE"]>=MASKRCNN_EVAL["threshold"]]
    yo=mask_overlay_panel(original,yp,color=(220,60,30),alpha=0.32)
    mo=mask_overlay_panel(original,mp,color=(40,100,230),alpha=0.32)
    yerr, ytp, yfp, yfn = error_code_overlay(original, yp, y["GROUND_TRUTH_INSTANCES"])
    merr, mtp, mfp, mfn = error_code_overlay(original, mp, m["GROUND_TRUTH_INSTANCES"])
    panels=[original,gt,yo,mo,yerr,merr]
    titles=["Original", "Ground truth", "YOLO26-L-seg", "Mask R-CNN",
            f"YOLO errors: TP {ytp} / FP {yfp} / FN {yfn}",
            f"Mask R-CNN errors: TP {mtp} / FP {mfp} / FN {mfn}"]
    fig,axes=plt.subplots(2,3,figsize=(19,11))
    for ax,img,title in zip(axes.flat,panels,titles):
        ax.imshow(img); ax.set_title(title); ax.axis("off")
    fig.suptitle(f"{frame_key} | error-map colors: TP=cyan, FP=red, FN=yellow")
    fig.tight_layout()
    out=FIGURE_DIR/f"qualitative_{suffix}_{frame_key}.png"
    fig.savefig(out,dpi=220,bbox_inches="tight"); fig.savefig(out.with_suffix(".pdf"),bbox_inches="tight")
    plt.close(fig)
    return out

# Deliberately diverse selection: maximum GT instance count, maximum cluster area, and top error frames per model.
frame_rank=[]
for key,rec in YOLO_BY_KEY.items():
    frame_rank.append({"FRAME_KEY":key,"SEQUENCE_ID":rec["SEQUENCE_ID"],"FRAME_ID":rec["FRAME_ID"],
                       "SPLIT":rec["SPLIT"],"GT_COUNT":len(rec["GROUND_TRUTH_INSTANCES"]),
                       "MAX_GT_AREA":max([x["AREA_PX"] for x in rec["GROUND_TRUTH_INSTANCES"]],default=0)})
frame_rank_df=pd.DataFrame(frame_rank)
chosen=[]
chosen += frame_rank_df.sort_values("GT_COUNT",ascending=False).head(2).FRAME_KEY.tolist()
chosen += frame_rank_df.sort_values("MAX_GT_AREA",ascending=False).head(2).FRAME_KEY.tolist()
for model_name in ["YOLO26-L-seg","Mask R-CNN + ConvNeXt-Tiny-FPN"]:
    errors=FRAME_ERRORS[FRAME_ERRORS.MODEL==model_name].sort_values(["TOTAL_ERRORS","FN","FP"],ascending=False)
    chosen += errors.FRAME_KEY.head(3).tolist()
# At least one frame per sequence, picked by the median frame index to show general appearance.
for seq in [1,2,3,4]:
    candidates=frame_rank_df[frame_rank_df.SEQUENCE_ID==seq].sort_values("FRAME_ID")
    chosen.append(candidates.iloc[len(candidates)//2].FRAME_KEY)
chosen=list(dict.fromkeys(chosen))[:MAX_QUALITATIVE_FRAMES]
qual_rows=[]
for i,key in enumerate(chosen,1):
    out=save_comparison_figure(key,f"panel{i:02d}")
    qual_rows.append({"FRAME_KEY":key,"FIGURE":str(out.relative_to(OUTPUT_DIR))})
pd.DataFrame(qual_rows).to_csv(OUTPUT_DIR/"qualitative_figure_index.csv",index=False)
print("Generated qualitative comparison panels:",len(qual_rows))

# Validation precision/recall threshold curves.
fig,ax=plt.subplots(figsize=(8,6))
for res in [YOLO_EVAL,MASKRCNN_EVAL]:
    sweep=res["sweep"].sort_values("confidence_threshold")
    ax.plot(sweep["recall"],sweep["precision"],label=res["model"])
ax.set_xlabel("Recall"); ax.set_ylabel("Precision"); ax.set_title("Validation precision-recall operating points"); ax.legend(); ax.grid(True,alpha=.25)
fig.tight_layout(); fig.savefig(FIGURE_DIR/"validation_precision_recall.png",dpi=220); fig.savefig(FIGURE_DIR/"validation_precision_recall.pdf"); plt.close(fig)

# Geometry accuracy scatter plots: only one-to-one mask-IoU>=0.50 matched test instances.
for field,title in [("AREA_PX","Area (px²)"),("WIDTH_PX","Width (px)"),("HEIGHT_PX","Height (px)")]:
    fig,ax=plt.subplots(figsize=(7,6))
    for model_name in MATCHED_DETAILS_DF.MODEL.unique():
        d=MATCHED_DETAILS_DF[(MATCHED_DETAILS_DF.MODEL==model_name)&(MATCHED_DETAILS_DF.SPLIT=="test")]
        ax.scatter(d[f"GT_{field}"],d[f"PRED_{field}"],s=9,alpha=.35,label=model_name)
    if not MATCHED_DETAILS_DF.empty:
        lim=max(float(MATCHED_DETAILS_DF.loc[MATCHED_DETAILS_DF.SPLIT=="test",f"GT_{field}"].max()),
                float(MATCHED_DETAILS_DF.loc[MATCHED_DETAILS_DF.SPLIT=="test",f"PRED_{field}"].max()))
        ax.plot([0,lim],[0,lim],"k--",linewidth=1)
    ax.set_xlabel(f"Ground-truth {title}"); ax.set_ylabel(f"Predicted {title}"); ax.set_title(f"Per-instance {title} agreement"); ax.legend(); ax.grid(True,alpha=.2)
    fig.tight_layout(); fig.savefig(FIGURE_DIR/f"geometry_scatter_{field.lower()}.png",dpi=220); fig.savefig(FIGURE_DIR/f"geometry_scatter_{field.lower()}.pdf"); plt.close(fig)

# Relative-size distributions are descriptive only; labels are per-frame observations.
fig,ax=plt.subplots(figsize=(9,5))
for split in ["val","test"]:
    d=MATURITY_DF[MATURITY_DF.SPLIT==split]
    for label,grp in d.groupby("ORIGINAL_CATEGORY_NAME"):
        ax.hist(grp.SIZE_INDEX_PCT,bins=35,alpha=.30,density=True,label=f"{split} / {label}")
ax.set_xlabel("Relative-size index (%)"); ax.set_ylabel("Density"); ax.set_title("Cluster area relative to training-only well-defined reference"); ax.legend(fontsize=7); ax.grid(True,alpha=.2)
fig.tight_layout(); fig.savefig(FIGURE_DIR/"relative_size_by_maturity_label.png",dpi=220); fig.savefig(FIGURE_DIR/"relative_size_by_maturity_label.pdf"); plt.close(fig)

Generated qualitative comparison panels: 12


In [18]:
TEST_TRACK=TRACKING_METRICS_DF[TRACKING_METRICS_DF.SPLIT=="test_retrospective"].copy()
fig,axes=plt.subplots(1,2,figsize=(13,5))
for ax,metric,title in zip(axes,["IDF1","MOTA_CUSTOM_FRAME_MATCH"],["IDF1","Custom MOTA"]):
    vals=TEST_TRACK.set_index("VARIANT")[metric]
    vals.plot(kind="bar",ax=ax); ax.set_title(title); ax.set_ylabel("Score"); ax.set_ylim(0,1); ax.tick_params(axis="x",rotation=20); ax.grid(axis="y",alpha=.2)
fig.suptitle("Retrospective 2×2 tracking comparison"); fig.tight_layout(); fig.savefig(FIGURE_DIR/"tracking_2x2_metrics.png",dpi=220); fig.savefig(FIGURE_DIR/"tracking_2x2_metrics.pdf"); plt.close(fig)

# Identify the most-observed source GT identities with one dominant predicted track in the selected C variant.
valid_hist=selected_obs.dropna(subset=["GT_TRACK_ID"]).copy() if "GT_TRACK_ID" in selected_obs.columns else pd.DataFrame()
track_figure_rows=[]
if not valid_hist.empty:
    counts=valid_hist.groupby(["SEQUENCE_ID","GT_TRACK_ID"]).agg(OBSERVATIONS=("FRAME_ID","nunique"),
        PRED_TRACKS=("PRED_TRACK_KEY","nunique")).reset_index()
    counts=counts[counts.PRED_TRACKS==1].sort_values("OBSERVATIONS",ascending=False)
    for rank,row in enumerate(counts.head(MAX_TRACK_GROWTH_FIGURES).itertuples(index=False),1):
        d=valid_hist[(valid_hist.SEQUENCE_ID==row.SEQUENCE_ID)&(valid_hist.GT_TRACK_ID==row.GT_TRACK_ID)].sort_values("FRAME_ID")
        fig,ax=plt.subplots(figsize=(8,4.5))
        ax.plot(d.ELAPSED_MINUTES_NOMINAL/60,d.AREA_PX,marker="o",markersize=3,label="Predicted tracked area")
        ax2=ax.twinx(); ax2.plot(d.ELAPSED_MINUTES_NOMINAL/60,d.SIZE_INDEX_PCT,color="tab:orange",alpha=.75,label="Relative-size index")
        ax.set_xlabel("Nominal elapsed time (hours)"); ax.set_ylabel("Mask area (px²)"); ax2.set_ylabel("Relative-size index (%)")
        ax.set_title(f"Sequence {row.SEQUENCE_ID} | source GT track {row.GT_TRACK_ID} | {len(d)} matched observations")
        ax.grid(True,alpha=.2)
        fig.tight_layout(); out=FIGURE_DIR/f"growth_seq{row.SEQUENCE_ID}_gt{row.GT_TRACK_ID}.png"; fig.savefig(out,dpi=220); fig.savefig(out.with_suffix(".pdf")); plt.close(fig)
        track_figure_rows.append({"SEQUENCE_ID":row.SEQUENCE_ID,"GT_TRACK_ID":row.GT_TRACK_ID,"OBSERVATIONS":len(d),"PRED_TRACKS":row.PRED_TRACKS,"FIGURE":out.name})
pd.DataFrame(track_figure_rows).to_csv(OUTPUT_DIR/"growth_figure_index.csv",index=False)

# Publication panel: the same source-verified cluster identity across time.
# Select an identity with multiple matched frames and one dominant predicted track.
if not valid_hist.empty:
    track_candidates=valid_hist.groupby(["SEQUENCE_ID","GT_TRACK_ID"]).agg(
        N_FRAMES=("FRAME_ID","nunique"),N_PRED_TRACKS=("PRED_TRACK_KEY","nunique")).reset_index()
    track_candidates=track_candidates[(track_candidates.N_FRAMES>=4)&(track_candidates.N_PRED_TRACKS==1)].sort_values("N_FRAMES",ascending=False)
    if not track_candidates.empty:
        chosen_track=track_candidates.iloc[0]
        d=valid_hist[(valid_hist.SEQUENCE_ID==chosen_track.SEQUENCE_ID)&
                     (valid_hist.GT_TRACK_ID==chosen_track.GT_TRACK_ID)].sort_values("FRAME_ID")
        pick_idx=np.unique(np.linspace(0,len(d)-1,min(4,len(d))).round().astype(int))
        sample_rows=d.iloc[pick_idx]
        fig,axes=plt.subplots(1,len(sample_rows),figsize=(5*len(sample_rows),5))
        if len(sample_rows)==1: axes=[axes]
        for ax,ob in zip(axes,sample_rows.itertuples(index=False)):
            frame=YOLO_BY_KEY[ob.FRAME_KEY]
            rgb=load_source_image_rgb(ob.FRAME_KEY)
            gt_inst=next(x for x in frame["GROUND_TRUTH_INSTANCES"] if str(x["GT_TRACK_ID"])==str(ob.GT_TRACK_ID))
            pred_inst=next(x for x in frame["PREDICTIONS"] if int(x["FRAME_INSTANCE_ID"])==int(ob.FRAME_INSTANCE_ID))
            panel=mask_overlay_panel(rgb,[gt_inst],color=(20,190,30),alpha=.34)
            panel=mask_overlay_panel(panel,[pred_inst],color=(220,70,30),alpha=.34)
            ax.imshow(panel); ax.set_title(f"Frame {int(ob.FRAME_ID)}\nArea {int(ob.AREA_PX):,} px²")
            ax.axis("off")
        fig.suptitle(f"Verified cluster progression | sequence {int(chosen_track.SEQUENCE_ID)} | GT track {chosen_track.GT_TRACK_ID}\nGreen=GT mask, red=tracked prediction")
        fig.tight_layout()
        out=FIGURE_DIR/f"tracking_identity_progression_seq{int(chosen_track.SEQUENCE_ID)}_gt{chosen_track.GT_TRACK_ID}.png"
        fig.savefig(out,dpi=220,bbox_inches="tight"); fig.savefig(out.with_suffix(".pdf"),bbox_inches="tight"); plt.close(fig)
        track_figure_rows.append({"SEQUENCE_ID":int(chosen_track.SEQUENCE_ID),"GT_TRACK_ID":str(chosen_track.GT_TRACK_ID),
                                  "OBSERVATIONS":int(chosen_track.N_FRAMES),"PRED_TRACKS":int(chosen_track.N_PRED_TRACKS),"FIGURE":out.name})
        pd.DataFrame(track_figure_rows).to_csv(OUTPUT_DIR/"growth_figure_index.csv",index=False)

# Sequence-wise throughput/runtime plots: model-forward time and postprocessing/export time reported separately.
RUNTIME_SUMMARY=runtime_df.groupby("MODEL").agg(FRAMES=("FRAME_KEY","size"),
    MEAN_INFERENCE_MS=("INFERENCE_MS","mean"),MEDIAN_INFERENCE_MS=("INFERENCE_MS","median"),
    P95_INFERENCE_MS=("INFERENCE_MS",lambda x:float(np.percentile(x,95))),
    MEAN_MASK_EXPORT_MS=("MASK_EXPORT_MS","mean"),PEAK_HOST_RSS_MB=("HOST_RSS_MB","max")).reset_index()
# GPU peak counters were reset once before the complete two-model inference run;
# these are whole-run peaks (including warm-up), not separate per-model peaks.
RUNTIME_SUMMARY["COMBINED_RUN_PEAK_CUDA_ALLOCATED_GIB"] = (torch.cuda.max_memory_allocated(0)/2**30) if RUNTIME_DETAILS_AVAILABLE else np.nan
RUNTIME_SUMMARY["COMBINED_RUN_PEAK_CUDA_RESERVED_GIB"] = (torch.cuda.max_memory_reserved(0)/2**30) if RUNTIME_DETAILS_AVAILABLE else np.nan
if not RUNTIME_DETAILS_AVAILABLE:
    RUNTIME_SUMMARY["RUNTIME_NOTE"] = "Inference timings were read from the previously completed fresh prediction records; host RSS and CUDA peak counters were not persisted before the earlier plotting failure."
RUNTIME_SUMMARY["GPU"] = torch.cuda.get_device_name(0)
RUNTIME_SUMMARY.to_csv(OUTPUT_DIR/"fresh_runtime_summary.csv",index=False)
fig,ax=plt.subplots(figsize=(8,5)); x=np.arange(len(RUNTIME_SUMMARY)); width=.36
ax.bar(x-width/2,RUNTIME_SUMMARY.MEAN_INFERENCE_MS,width,label="Model inference")
ax.bar(x+width/2,RUNTIME_SUMMARY.MEAN_MASK_EXPORT_MS,width,label="Mask export / geometry / RLE")
ax.set_xticks(x,RUNTIME_SUMMARY.MODEL,rotation=15); ax.set_ylabel("Milliseconds per frame"); ax.set_title("Fresh runtime breakdown (timing boundaries differ by model)"); ax.legend(); ax.grid(axis="y",alpha=.2)
fig.tight_layout(); fig.savefig(FIGURE_DIR/"runtime_breakdown.png",dpi=220); fig.savefig(FIGURE_DIR/"runtime_breakdown.pdf"); plt.close(fig)

display(RUNTIME_SUMMARY)

# Remove only temporary source archives extracted again for figures; attached source data is untouched.
cleanup_visual_source_extractions()
for seq in [1, 2, 3, 4]:
    if SEQUENCE_DATA[seq]["source_mode"] in {"attached_archive", "google_drive"}:
        shutil.rmtree(SEQUENCE_DATA[seq]["root"], ignore_errors=True)


,MODEL,FRAMES,MEAN_INFERENCE_MS,MEDIAN_INFERENCE_MS,P95_INFERENCE_MS,MEAN_MASK_EXPORT_MS,PEAK_HOST_RSS_MB,COMBINED_RUN_PEAK_CUDA_ALLOCATED_GIB,COMBINED_RUN_PEAK_CUDA_RESERVED_GIB,GPU
0,Mask R-CNN + ConvNeXt-Tiny-FPN,961,344.409694,353.157685,386.096008,5085.541984,2627.886719,1.034461,1.248047,Tesla T4
1,YOLO26-L-seg,961,156.087631,157.048607,161.143592,1258.915700,2551.863281,1.034461,1.248047,Tesla T4


In [19]:
rng=np.random.default_rng(RANDOM_SEED)
max_cost_diff=0.0; matrices_tested=0; categories=Counter()
for _ in range(500):
    n=int(rng.integers(1,13)); m=int(rng.integers(1,13)); cost=rng.normal(size=(n,m))
    rr,cc=hungarian_min_cost(cost); sr,sc=linear_sum_assignment(cost)
    diff=abs(float(cost[rr,cc].sum())-float(cost[sr,sc].sum()))
    max_cost_diff=max(max_cost_diff,diff); matrices_tested+=1; categories["random_rectangular"]+=1
    assert len(rr)==min(n,m) and len(set(rr.tolist()))==len(rr) and len(set(cc.tolist()))==len(cc)
for shape in [(2,5),(5,2),(4,4),(1,7),(7,1)]:
    cost=rng.integers(0,4,size=shape).astype(float)
    rr,cc=hungarian_min_cost(cost); sr,sc=linear_sum_assignment(cost)
    max_cost_diff=max(max_cost_diff,abs(float(cost[rr,cc].sum())-float(cost[sr,sc].sum())))
    matrices_tested+=1; categories["integer_ties"]+=1
for shape in [(0,0),(0,4),(5,0)]:
    rr,cc=hungarian_min_cost(np.empty(shape)); assert len(rr)==len(cc)==0
    matrices_tested+=1; categories["empty_dimensions"]+=1
sample=np.array([[.90,.80],[.85,0.]])
pairs=max_cardinality_iou_matches(sample,.50)
assert len(pairs)==2 and {(p,g) for p,g,_ in pairs}=={(0,1),(1,0)}
hungarian_report={"algorithm":"from-scratch rectangular Hungarian shortest-augmenting-path solver",
                  "reference":"scipy.optimize.linear_sum_assignment","seed":RANDOM_SEED,"matrices_tested":matrices_tested,
                  "categories":dict(categories),"maximum_absolute_total_cost_difference":max_cost_diff,
                  "empty_dimensions":"PASS","rectangular_assignments":"PASS","tie_cases":"objective values compared; pair ordering may differ",
                  "maximum_cardinality_iou_case":"PASS","result":"PASS" if max_cost_diff<1e-8 else "FAIL"}
(OUTPUT_DIR/"hungarian_verification_fresh.json").write_text(json.dumps(hungarian_report,indent=2),encoding="utf-8")
print(json.dumps(hungarian_report,indent=2))
assert hungarian_report["result"]=="PASS"

{
  "algorithm": "from-scratch rectangular Hungarian shortest-augmenting-path solver",
  "reference": "scipy.optimize.linear_sum_assignment",
  "seed": 42,
  "matrices_tested": 508,
  "categories": {
    "random_rectangular": 500,
    "integer_ties": 5,
    "empty_dimensions": 3
  },
  "maximum_absolute_total_cost_difference": 0.0,
  "empty_dimensions": "PASS",
  "rectangular_assignments": "PASS",
  "tie_cases": "objective values compared; pair ordering may differ",
  "maximum_cardinality_iou_case": "PASS",
  "result": "PASS"
}


## Reproducibility record

The final cell reconciles fresh metrics with the previous evidence, records the environment and provenance, writes the integrity report, and packages the predictions, tables, figures, and run manifest. Differences from prior results are reported rather than silently overwritten.

In [20]:
# Check canonical dimensions and source identity mapping one last time.
frame_df=pd.DataFrame(frame_audit_rows)
checks={
    "manifest_sha256_matches_expected": MANIFEST_SHA256==EXPECTED_MANIFEST_SHA256,
    "yolo_checkpoint_sha256_matches_expected": YOLO_SHA256==EXPECTED_YOLO_SHA256,
    "maskrcnn_checkpoint_sha256_matches_expected": MASKRCNN_SHA256==EXPECTED_MASKRCNN_SHA256,
    "ultralytics_version_matches": ultralytics.__version__==ULTRALYTICS_VERSION,
    "fresh_yolo_frame_count_961": len(YOLO_RECORDS)==961,
    "fresh_maskrcnn_frame_count_961": len(MASKRCNN_RECORDS)==961,
    "fresh_record_split_labels_match_manifest": all(
        rec.get("SPLIT") == expected_split_by_key.get(rec.get("FRAME_KEY"))
        for rec in YOLO_RECORDS + MASKRCNN_RECORDS
    ),
    "tracking_ablation_has_eight_unique_variant_split_rows": (
        len(TRACKING_METRICS_DF) == 8
        and not TRACKING_METRICS_DF.duplicated(["VARIANT", "SPLIT"]).any()
    ),
    "selected_tracker_history_export_exists": (OUTPUT_DIR / "selected_yolo_growth_histories_with_source_ids.csv").is_file(),
    "source_image_integrity_rows_961": len(frame_integrity_df)==961 and frame_integrity_df.FRAME_KEY.nunique()==961,
    "source_image_hashes_valid": bool(frame_integrity_df.SOURCE_IMAGE_SHA256.astype(str).str.fullmatch(r"[0-9a-f]{64}").all()),
    "runtime_rows_1922": len(runtime_df)==1922,
    "runtime_frames_per_model_961": runtime_df.groupby("MODEL").FRAME_KEY.nunique().to_dict()=={"YOLO26-L-seg":961,"Mask R-CNN + ConvNeXt-Tiny-FPN":961},
    "fresh_record_image_hashes_match_source_ledger": all(
        rec.get("SOURCE_IMAGE_SHA256")==source_hashes.get(rec["FRAME_KEY"])
        for rec in YOLO_RECORDS+MASKRCNN_RECORDS
    ),
    "evaluation_identity_rows_6084": len(IDENTITY_MAP_DF)==6084,
    "all_gt_track_ids_present": bool(IDENTITY_MAP_DF.GT_TRACK_ID.notna().all()),
    "full_source_ids_per_sequence_match_prior_audit": {s: len(SEQUENCE_DATA[s]["source_track_ids"]) for s in [1,2,3,4]} == EXPECTED_FULL_SOURCE_TRACK_IDS,
    "eval_subset_ids_per_sequence_match_manifest": IDENTITY_MAP_DF.groupby("SEQUENCE_ID").GT_TRACK_ID.nunique().to_dict() == EXPECTED_EVAL_SUBSET_TRACK_IDS,
    "eval_ids_are_subset_of_full_source_ids": all(
        set(IDENTITY_MAP_DF.loc[IDENTITY_MAP_DF.SEQUENCE_ID == s, "GT_TRACK_ID"].astype(str)).issubset(set(SEQUENCE_DATA[s]["source_track_ids"]))
        for s in [1,2,3,4]
    ),
    "training_reference_count_matches": len(reference_df)==3365 and len(reference_frames)==644,
    "training_reference_mean_matches": bool(np.isclose(A_REF_PX2,669444.1598811293,atol=.5)),
    "hungarian_verification_passes": hungarian_report["result"]=="PASS",
    "fresh_rle_geometry_checks_passed": bool(RLE_GEOMETRY_CHECKS_PASSED),
    "maskrcnn_cpu_paste_matches_torchvision": bool(MASKRCNN_CPU_PASTE_PARITY),
}

# Historical numerical comparisons are diagnostics. A REVIEW status never silently changes the result.
reproduction_all_match=bool((REPRODUCTION_AUDIT_DF.STATUS=="MATCH").all())
checks["historical_segmentation_and_geometry_metrics_within_tolerance"]=reproduction_all_match

tracking_expected={
 "A_iou_only":{"IDF1":0.7430066412,"MOTA_CUSTOM_FRAME_MATCH":0.8328952309,"IDSW":56,"PREDICTED_TRACKS":83},
 "B_spatial_only":{"IDF1":0.7647413967,"MOTA_CUSTOM_FRAME_MATCH":0.8302666166,"IDSW":63,"PREDICTED_TRACKS":91},
 "C_iou_plus_lowconf":{"IDF1":0.8556338028,"MOTA_CUSTOM_FRAME_MATCH":0.8918512955,"IDSW":6,"PREDICTED_TRACKS":30},
 "D_spatial_plus_lowconf":{"IDF1":0.8587084149,"MOTA_CUSTOM_FRAME_MATCH":0.8877206158,"IDSW":15,"PREDICTED_TRACKS":41},
}
tracking_audit=[]
for variant, vals in tracking_expected.items():
    variant_rows = TRACKING_METRICS_DF[
        (TRACKING_METRICS_DF.SPLIT == "test_retrospective")
        & (TRACKING_METRICS_DF.VARIANT == variant)
    ]
    if len(variant_rows) != 1:
        raise RuntimeError(
            f"Expected exactly one retrospective test row for tracker {variant!r}; "
            f"found {len(variant_rows)}. Available variants: "
            f"{TRACKING_METRICS_DF[TRACKING_METRICS_DF.SPLIT == 'test_retrospective'].VARIANT.tolist()}"
        )
    row = variant_rows.iloc[0]
    for metric, old in vals.items():
        fresh=row[metric]; tol=0 if metric in {"IDSW","PREDICTED_TRACKS"} else .005
        tracking_audit.append({"VARIANT":variant,"METRIC":metric,"PRIOR_VALUE":old,"FRESH_VALUE":float(fresh),
                               "ABS_DIFFERENCE":abs(float(fresh)-float(old)),"TOLERANCE":tol,
                               "STATUS":"MATCH" if abs(float(fresh)-float(old))<=tol else "REVIEW"})
TRACKING_REPRO_AUDIT_DF=pd.DataFrame(tracking_audit)
TRACKING_REPRO_AUDIT_DF.to_csv(OUTPUT_DIR/"prior_tracking_reproduction_audit.csv",index=False)
checks["historical_tracking_metrics_within_tolerance"]=bool((TRACKING_REPRO_AUDIT_DF.STATUS=="MATCH").all())

integrity_report={"created_utc":datetime.now(timezone.utc).isoformat(),"checks":checks,
                  "overall_status":"PASS" if all(checks.values()) else "REVIEW_REQUIRED",
                  "notes":[
                    ("Fresh inference was executed from both exact checkpoints in this notebook execution." if not INFERENCE_CACHE_REUSED else "Previously completed fresh inference records were revalidated against both checkpoint hashes, all 961 frame keys, and source-image hashes; this execution resumed post-inference analysis and figure export."),
                    "The test split was inspected in earlier development. Fresh recomputation does not make test results untouched or independent.",
                    "Custom MOTA is the project's frame-matching measure, not standard CLEAR-MOTA/TrackEval.",
                    "Raw timestamps are reported only when present in source annotation metadata; otherwise time is nominal.",
                    "The notebook is the reproducible execution artifact; no separate application demo is claimed."
                  ]}
(OUTPUT_DIR/"integrity_report.json").write_text(json.dumps(integrity_report,indent=2),encoding="utf-8")

run_manifest={
 "created_utc":datetime.now(timezone.utc).isoformat(),
 "notebook":"full-inference.ipynb",
 "inference_source":("fresh model forward passes in this execution" if not INFERENCE_CACHE_REUSED else "validated records from a previously completed fresh-inference pass; reused only to resume post-inference analysis after a figure-export failure"),
 "fresh_inference_performed_this_execution":not INFERENCE_CACHE_REUSED,
 "completed_fresh_prediction_records_reused":INFERENCE_CACHE_REUSED,
 "maskrcnn_mask_pasting":"TorchVision paste_masks_in_image applied one ROI at a time on CPU; model ROI inference and source-coordinate box resizing unchanged",
 "maskrcnn_cpu_paste_parity_test":"PASS" if MASKRCNN_CPU_PASTE_PARITY else "FAIL",
 "yolo_inference_implementation":"Supplied fungicirno-yolo26-validation-test-inference-cache.ipynb settings: imgsz=1024, conf=0.001, iou=0.70, nms=None, max_det=1000, retina_masks=True, batch=1",
 "manifest_path":str(MANIFEST_PATH),"manifest_sha256":MANIFEST_SHA256,
 "yolo_checkpoint_path":str(YOLO_MODEL_PATH),"yolo_checkpoint_sha256":YOLO_SHA256,
 "maskrcnn_checkpoint_path":str(MASKRCNN_CHECKPOINT_PATH),"maskrcnn_checkpoint_sha256":MASKRCNN_SHA256,
 "maskrcnn_model_config_path":str(MASKRCNN_CONFIG_PATH) if MASKRCNN_CONFIG_PATH else None,
 "maskrcnn_class_mapping_path":str(MASKRCNN_CLASS_MAP_PATH) if MASKRCNN_CLASS_MAP_PATH else None,
 "maskrcnn_model_config_sha256":sha256_file(MASKRCNN_CONFIG_PATH) if MASKRCNN_CONFIG_PATH and MASKRCNN_CONFIG_PATH.is_file() else None,
 "maskrcnn_class_mapping_sha256":sha256_file(MASKRCNN_CLASS_MAP_PATH) if MASKRCNN_CLASS_MAP_PATH and MASKRCNN_CLASS_MAP_PATH.is_file() else None,
 "source_annotation_files":{str(s):str(SEQUENCE_DATA[s]["tracked_json_path"]) for s in [1,2,3,4]},
 "source_annotation_sha256":{str(s):SEQUENCE_DATA[s]["tracked_json_sha256"] for s in [1,2,3,4]},
 "source_image_hashes_csv":"fresh_frame_integrity.csv (one SHA-256 per evaluated source image)",
 "source_track_id_scope_audit_csv":"source_tracking_id_scope_audit.csv (full-source versus eval-subset identity counts)",
 "source_archive_sha256":{str(s):SEQUENCE_DATA[s]["source_archive_sha256"] for s in [1,2,3,4]},
 "model_settings":{"yolo_imgsz":YOLO_IMAGE_SIZE,"yolo_confidence_floor":YOLO_SCORE_FLOOR,"yolo_nms_iou":YOLO_NMS_IOU,
                   "yolo_nms_mode":YOLO_NMS_MODE,"yolo_max_det":YOLO_MAX_DET,"yolo_retina_masks":YOLO_RETINA_MASKS,
                   "maskrcnn_score_floor":MASKRCNN_SCORE_FLOOR,"maskrcnn_max_det":MASKRCNN_MAX_DET,
                   "maskrcnn_mask_threshold":MASKRCNN_MASK_THRESHOLD,
                   "maskrcnn_full_resolution_pasting":"CPU per-instance via torchvision paste_masks_in_image"},
 "selected_validation_thresholds":{"YOLO26-L-seg":YOLO_EVAL["threshold"],"Mask R-CNN + ConvNeXt-Tiny-FPN":MASKRCNN_EVAL["threshold"]},
 "tracker":{"association_iou":TRACKER_ASSOCIATION_IOU,"low_confidence_min":TRACKER_LOW_CONFIDENCE_MIN,
            "max_gap":TRACKER_MAX_GAP,"selected_history_variant":SELECTED_TRACKING_VARIANT,
            "spatial_distance_px":TRACKER_SPATIAL_DISTANCE_PX,
            "high_confidence_threshold":YOLO_EVAL["threshold"],"validation_stride_offsets":list(range(VALIDATION_CADENCE_STRIDE))},
 "reference_size_area_px2":A_REF_PX2,
 "environment":{"python":sys.version,"platform":platform.platform(),"torch":torch.__version__,
                "torchvision":torchvision.__version__,"cuda":torch.version.cuda,"gpu":torch.cuda.get_device_name(0),
                "ultralytics":ultralytics.__version__,"numpy":np.__version__,"pandas":pd.__version__,"opencv":cv2.__version__},
 "runtime":{"combined_run_cuda_peak_allocated_gib":(torch.cuda.max_memory_allocated(0)/2**30) if RUNTIME_DETAILS_AVAILABLE else None,
            "combined_run_cuda_peak_reserved_gib":(torch.cuda.max_memory_reserved(0)/2**30) if RUNTIME_DETAILS_AVAILABLE else None,
            "gpu_peak_scope":"whole two-model run including warm-up; not per-model" if RUNTIME_DETAILS_AVAILABLE else "not captured in this cache-resume execution"},
 "timing_boundaries":{"yolo":"synchronized yolo_model.predict call, including Ultralytics preprocessing/postprocessing and mask generation",
                      "mask_rcnn":"synchronized model forward plus TorchVision ROI/box postprocess; image decode, RGB conversion, tensor creation and device transfer are outside the timer",
                      "mask_export":"per-instance full-resolution CPU mask paste (Mask R-CNN), geometry and RLE export; not identical postprocessing work across models",
                      "comparability":"descriptive timing evidence only; not a controlled model-speed comparison"},
 "frame_counts":{"train":int((manifest.split=="train").sum()),"val":int((manifest.split=="val").sum()),"test":int((manifest.split=="test").sum()),
                 "fresh_inference_frames_per_model":961},
 "track_identity_scope":{"full_source_unique_ids_by_sequence":{str(s):len(SEQUENCE_DATA[s]["source_track_ids"]) for s in [1,2,3,4]},
                         "eval_subset_unique_ids_by_sequence":{str(s):int(IDENTITY_MAP_DF[IDENTITY_MAP_DF.SEQUENCE_ID==s].GT_TRACK_ID.nunique()) for s in [1,2,3,4]}},
 "fresh_cache_files":[str(YOLO_CACHE_PATH.relative_to(OUTPUT_DIR)),str(MASKRCNN_CACHE_PATH.relative_to(OUTPUT_DIR))],
 "integrity_status":integrity_report["overall_status"]
}
(OUTPUT_DIR/"run_manifest.json").write_text(json.dumps(run_manifest,indent=2),encoding="utf-8")

readme="""# Oyster Mushroom Cluster Monitoring — inference evidence bundle\n\nThis bundle was generated by `full-inference.ipynb`. The run manifest states whether model inference ran in this execution or completed fresh-inference records were validated and reused to resume post-inference analysis after an interruption. The YOLO26 inference arguments match the previously supplied YOLO inference notebook. Mask R-CNN uses the project's established memory-safe postprocess: source-coordinate boxes are resized by TorchVision and each full-resolution ROI mask is pasted individually on CPU with TorchVision's `paste_masks_in_image`, avoiding the default multi-GiB GPU mask stack.\n\n## Core contents\n- `fresh_prediction_records/`: fresh full-resolution YOLO26 and Mask R-CNN prediction records with compressed COCO-RLE masks and source GT identity fields.\n- `fresh_segmentation_metrics.csv`, `fresh_validation_threshold_sweeps.csv`, `fresh_per_instance_geometry_metrics.csv`, `fresh_matched_instance_details.csv`: common-protocol benchmark results.\n- `fresh_tracking_2x2_metrics.csv` and per-variant histories: re-evaluated frozen tracker comparisons; test results remain retrospective.\n- `fresh_source_track_id_lookup.csv` and `source_tracking_id_scope_audit.csv`: per-observation source IDs plus a full-source-versus-evaluation-subset count audit (43 full-source IDs; 40 observed in the val/test subset).\n- `well_defined_reference_size.json` and `hungarian_verification_fresh.json`: relative-size reference and assignment verification evidence.\n- `figures/`: six-panel original/GT/model-overlay/error maps, time-series charts, geometry agreement, PR, relative-size, runtime, ablation, and source-ID-linked cluster progression plots in PNG/PDF formats.\n- `run_manifest.json`, `environment.json`, `pip_freeze.txt`, `integrity_report.json`, and per-image/annotation/checkpoint hashes: provenance and reproducibility evidence. In cache-resume mode, host RSS and whole-run CUDA peak counters are marked unavailable rather than inferred. If a completed fresh inference cache is reused after a plotting/export interruption, the manifest records this explicitly; inference is never silently claimed to have run again.\n\n## Important limitations\n- The source test split has been seen during earlier development; these results are re-computations, not an untouched test.\n- `MOTA_CUSTOM_FRAME_MATCH` is not standard TrackEval/CLEAR MOTA.\n- The relative-size index is not a maturity classifier.\n- The notebook is the reproducible execution artifact; no separate application demo is claimed.\n"""
(OUTPUT_DIR/"README_EVIDENCE_BUNDLE.md").write_text(readme,encoding="utf-8")

# Save environment and all numerical summaries in machine-readable form.
(OUTPUT_DIR/"environment.json").write_text(json.dumps(run_manifest["environment"],indent=2),encoding="utf-8")
(OUTPUT_DIR/"pip_freeze.txt").write_text(subprocess.check_output([sys.executable,"-m","pip","freeze"],text=True),encoding="utf-8")
SEGMENTATION_METRICS_DF.to_csv(OUTPUT_DIR/"fresh_segmentation_metrics.csv",index=False)
TRACKING_METRICS_DF.to_csv(OUTPUT_DIR/"fresh_tracking_2x2_metrics.csv",index=False)

# Include every evidence file. Compression level 4 balances packaging time and size for RLE data/images.
BUNDLE_PATH=Path("/kaggle/working/Oyster_Mushroom_Cluster_Monitoring_Independent_Evidence_Bundle.zip")
BUNDLE_PATH.unlink(missing_ok=True)
with zipfile.ZipFile(BUNDLE_PATH,"w",compression=zipfile.ZIP_DEFLATED,compresslevel=4) as zf:
    for path in OUTPUT_DIR.rglob("*"):
        if path.is_file(): zf.write(path,arcname=path.relative_to(OUTPUT_DIR))
print("Evidence status:",integrity_report["overall_status"])
print("Evidence bundle:",BUNDLE_PATH,"size MiB:",round(BUNDLE_PATH.stat().st_size/2**20,2))
display(pd.DataFrame([{"CHECK":k,"STATUS":"PASS" if v else "REVIEW"} for k,v in checks.items()]))

Evidence status: PASS
Evidence bundle: /kaggle/working/FungiCirno_Independent_Evidence_Bundle.zip size MiB: 149.35


,CHECK,STATUS
0,manifest_sha256_matches_expected,PASS
1,yolo_checkpoint_sha256_matches_expected,PASS
2,maskrcnn_checkpoint_sha256_matches_expected,PASS
3,ultralytics_version_matches,PASS
4,fresh_yolo_frame_count_961,PASS
5,fresh_maskrcnn_frame_count_961,PASS
6,fresh_record_split_labels_match_manifest,PASS
7,tracking_ablation_has_eight_unique_variant_spl...,PASS
8,selected_tracker_history_export_exists,PASS
9,source_image_integrity_rows_961,PASS
